# code-server on Colab — plug and play

**Runtime → Run all.** No clone, no Drive, no patch. The only optional input is
an ngrok token; without one it falls back to a Cloudflare quick tunnel that needs
no account, so a bare run works.

What it sets up: `code-server` on `127.0.0.1:8080`, password auth, a public
tunnel, and a supervisor that restarts on crash and shuts the VM down when idle.

Nothing here renames processes or spoofs traffic — that layer is not what Colab
enforces on, and the effort buys nothing. See `README.md` (written to disk in the
next cells) for the layer-by-layer breakdown. What this *does* fix: rotating
URLs, blank pages, and VMs held open with no one using them.

Colab's FAQ does not list remote-IDE use as supported. It works; the runtime is
ephemeral and account-level throttling is always possible.

## 1. Environment

In [ ]:
import os, sys, platform, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules or pathlib.Path("/content").is_dir()
BASE = pathlib.Path("/content/colab-cs")

print("colab      :", IN_COLAB)
print("platform   :", platform.platform())
print("machine    :", platform.machine())
print("python     :", platform.python_version())
try:
    print("free disk  :", shutil.disk_usage("/content").free // 2**30, "GiB")
except OSError:
    pass
print("install to :", BASE)

## 2. Configuration

Every value is optional. `TUNNEL` stays `auto` unless you want to force one:

* `auto` — ngrok if it has credentials, else a cloudflared quick tunnel
* `ngrok` — needs `NGROK_TOKEN` (or a Colab Secret named `NGROK_TOKEN`)
* `cloudflared` — no account, but the URL changes every run
* `none` — localhost only, for debugging

In [ ]:
import os

# ── edit here if you want ─────────────────────────────────────────────────────
NGROK_TOKEN    = ""        # or add a Colab Secret named NGROK_TOKEN
TUNNEL         = ""        # "" = auto | "ngrok" | "cloudflared" | "none"
NGROK_DOMAIN   = ""        # reserved domain => stable URL, e.g. "me.ngrok-free.app"
NGROK_AUTH     = ""        # optional edge gate "user:pass" (rejects scanners)
IDLE_TIMEOUT   = 1800      # seconds with zero connections before shutdown
MAX_RUNTIME    = 39600     # seconds before forced shutdown (11 h)
STATE_ON_DRIVE = False     # keep extensions and settings between runtimes
CS_EXTENSIONS  = ""        # comma-separated IDs, installed once
# ─────────────────────────────────────────────────────────────────────────────

token = NGROK_TOKEN.strip()
token_source = "config cell" if token else ""
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("NGROK_TOKEN") or "").strip()
        token_source = "Colab secret" if token else ""
    except Exception:
        pass

os.environ["NGROK_TOKEN"] = token
os.environ["CS_TUNNEL"] = TUNNEL
os.environ["CS_IDLE_TIMEOUT"] = str(IDLE_TIMEOUT)
os.environ["CS_MAX_RUNTIME"] = str(MAX_RUNTIME)
if NGROK_DOMAIN: os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN
if NGROK_AUTH:   os.environ["NGROK_BASIC_AUTH"] = NGROK_AUTH
if CS_EXTENSIONS: os.environ["CS_EXTENSIONS"] = CS_EXTENSIONS

if STATE_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CS_STATE_DIR"] = "/content/drive/MyDrive/.code-server"
    os.environ["CS_CACHE_DIR"] = "/content/drive/MyDrive/.code-server/cache"

print("tunnel      :", TUNNEL or "auto")
print("ngrok token :", f"yes ({token_source})" if token else "no -> cloudflared fallback")
print("state dir   :", os.environ.get("CS_STATE_DIR", "/content/.code-server"))

## 3. Write the toolkit

In [ ]:
import base64, pathlib, stat

FILES = {
    'lib/common.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojIGNvbGFiL2xpYi9jb21tb24uc2gg4oCUIHNoYXJlZCBoZWxwZXJzIGZvciB0aGUgQ29sYWIgc2NyaXB0cy4KIwojIFNvdXJjZSB0aGlzIGZpbGUsIGRvIG5vdCBleGVjdXRlIGl0OgojICAgc291cmNlICIkKGRpcm5hbWUgIiQwIikvbGliL2NvbW1vbi5zaCIKIwojIEV2ZXJ5dGhpbmcgaGVyZSBpcyBkZWxpYmVyYXRlbHkgc2lkZS1lZmZlY3QgZnJlZSBhcGFydCBmcm9tIGluc3RhbGxpbmcgYW4gRVJSCiMgdHJhcCBhbmQgdGhlIGNvbG91ciBjb25zdGFudHMuCgppZiBbWyAtbiAiJHtDU19DT01NT05fTE9BREVEOi19IiBdXTsgdGhlbgogIHJldHVybiAwCmZpCkNTX0NPTU1PTl9MT0FERUQ9MQoKc2V0IC1FZXVvIHBpcGVmYWlsCgojIHNoZWxsY2hlY2sgZGlzYWJsZT1TQzIxMjggICMgQkFTSF9TT1VSQ0UgaXMgdGhlIGZpbGUgYmVpbmcgc291cmNlZApyZWFkb25seSBDU19MSUJfRElSPSIkKGNkIC0tICIkKGRpcm5hbWUgLS0gIiR7QkFTSF9TT1VSQ0VbMF19IikiICYmIHB3ZCAtUCkiCnJlYWRvbmx5IENTX1JFUE9fUk9PVD0iJChjZCAtLSAiJHtDU19MSUJfRElSfS8uLi8uLiIgJiYgcHdkIC1QKSIKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG91dHB1dCAtLS0tCmlmIFtbIC10IDEgXV07IHRoZW4KICBDU19DX1JFU0VUPSQnXDAzM1swbScKICBDU19DX0JMVUU9JCdcMDMzWzE7MzRtJwogIENTX0NfR1JFRU49JCdcMDMzWzE7MzJtJwogIENTX0NfUkVEPSQnXDAzM1sxOzMxbScKICBDU19DX1lFTExPVz0kJ1wwMzNbMTszM20nCiAgQ1NfQ19ESU09JCdcMDMzWzJtJwplbHNlCiAgQ1NfQ19SRVNFVD0nJyBDU19DX0JMVUU9JycgQ1NfQ19HUkVFTj0nJyBDU19DX1JFRD0nJyBDU19DX1lFTExPVz0nJyBDU19DX0RJTT0nJwpmaQoKbG9nKCkgIHsgcHJpbnRmICclcz09PiVzICVzXG4nICAgIiRDU19DX0JMVUUiICAgIiRDU19DX1JFU0VUIiAiJCoiOyB9Cm9rKCkgICB7IHByaW50ZiAnJXMgb2slcyAlc1xuJyAgICIkQ1NfQ19HUkVFTiIgICIkQ1NfQ19SRVNFVCIgIiQqIjsgfQp3YXJuKCkgeyBwcmludGYgJyVzd2FybiVzICVzXG4nICAiJENTX0NfWUVMTE9XIiAiJENTX0NfUkVTRVQiICIkKiIgPiYyOyB9CmRpZSgpICB7IHByaW50ZiAnJXNlcnIlcyAlc1xuJyAgICIkQ1NfQ19SRUQiICAgICIkQ1NfQ19SRVNFVCIgIiQqIiA+JjI7IGV4aXQgMTsgfQpkaW0oKSAgeyBwcmludGYgJyVzJXMlc1xuJyAgICAgICAiJENTX0NfRElNIiAgICAiJCoiICIkQ1NfQ19SRVNFVCI7IH0KCiMgUmVwb3J0IHRoZSBmYWlsaW5nIGxpbmUgaW5zdGVhZCBvZiBkeWluZyBzaWxlbnRseS4gRXhlbXB0IGZyb20gdGhlIHVzdWFsCiMgYCYmYC9gfHxgL2BpZmAgY2FzZXMsIHNvIGl0IG9ubHkgZmlyZXMgb24gZ2VudWluZWx5IHVuZXhwZWN0ZWQgZmFpbHVyZXMuCmNzX29uX2VycigpIHsKICBsb2NhbCBlYz0kPyBsaW5lPSQxCiAgcHJpbnRmICclc2VyciVzICVzOiVzIGV4aXRlZCB3aXRoICVkXG4nIFwKICAgICIkQ1NfQ19SRUQiICIkQ1NfQ19SRVNFVCIgIiQoYmFzZW5hbWUgLS0gIiR7QkFTSF9TT1VSQ0VbMl06LT99IikiICIkbGluZSIgIiRlYyIgPiYyCiAgZXhpdCAiJGVjIgp9CnRyYXAgJ2NzX29uX2VyciAiJExJTkVOTyInIEVSUgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3lzdGVtIC0tLS0KY3NfaXNfcm9vdCgpIHsgW1sgIiQoaWQgLXUpIiAtZXEgMCBdXTsgfQoKY3Nfcm9vdCgpIHsKICBpZiBjc19pc19yb290OyB0aGVuICIkQCI7IGVsc2Ugc3VkbyAtLSAiJEAiOyBmaQp9Cgpjc19uZWVkKCkgewogIGxvY2FsIGNtZD0kMSBoaW50PSR7MjotfQogIGNvbW1hbmQgLXYgIiRjbWQiID4vZGV2L251bGwgMj4mMSB8fCBkaWUgIm1pc3NpbmcgY29tbWFuZDogJHtjbWR9JHtoaW50OisgKCR7aGludH0pfSIKfQoKY3Nfb3NfZmllbGQoKSB7CiAgKCAjIHNoZWxsY2hlY2sgZGlzYWJsZT1TQzEwOTEKICAgIC4gL2V0Yy9vcy1yZWxlYXNlIDI+L2Rldi9udWxsIHx8IGV4aXQgMAogICAgcHJpbnRmICclcycgIiR7ITE6LX0iCiAgKQp9CgojIFVidW50dSAyNC4wNCsgcmVuYW1lZCBsaWJhc291bmQyIC0+IGxpYmFzb3VuZDJ0NjQuIENvbGFiIHN0aWxsIHNoaXBzIDIyLjA0LAojIGJ1dCB0aGUgaW1hZ2UgY2hhbmdlcyB3aXRob3V0IG5vdGljZSwgc28gZGV0ZWN0IGluc3RlYWQgb2YgaGFyZGNvZGluZy4KY3NfYXNvdW5kX3BrZygpIHsKICBjYXNlICIkKGNzX29zX2ZpZWxkIFZFUlNJT05fSUQpIiBpbgogICAgMjQuMDQqfDI1Lip8MjYuKnwyWzctOV0uKikgcHJpbnRmICdsaWJhc291bmQydDY0JyA7OwogICAgKikgICAgICAgICAgICAgICAgICAgICAgICAgcHJpbnRmICdsaWJhc291bmQyJyA7OwogIGVzYWMKfQoKY3NfYXJjaCgpIHsKICBjYXNlICIkKHVuYW1lIC1tKSIgaW4KICAgIHg4Nl82NHxhbWQ2NCkgIHByaW50ZiAnYW1kNjQnIDs7CiAgICBhYXJjaDY0fGFybTY0KSBwcmludGYgJ2FybTY0JyA7OwogICAgKikgZGllICJ1bnN1cHBvcnRlZCBhcmNoaXRlY3R1cmU6ICQodW5hbWUgLW0pIiA7OwogIGVzYWMKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RhdGUgZGlyIC0tLS0KY3Nfc3RhdGVfZGlyKCkgewogIGxvY2FsIGQ9IiR7Q1NfU1RBVEVfRElSOi0vY29udGVudC8uY29kZS1zZXJ2ZXJ9IgogIG1rZGlyIC1wIC0tICIkZCIgfHwgZGllICJjYW5ub3QgY3JlYXRlIHN0YXRlIGRpcjogJHtkfSIKICBjaG1vZCA3MDAgLS0gIiRkIiAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgcHJpbnRmICclcycgIiRkIgp9Cgpjc19jYWNoZV9kaXIoKSB7CiAgbG9jYWwgZD0iJHtDU19DQUNIRV9ESVI6LSQoY3Nfc3RhdGVfZGlyKS9jYWNoZX0iCiAgbWtkaXIgLXAgLS0gIiRkIiB8fCBkaWUgImNhbm5vdCBjcmVhdGUgY2FjaGUgZGlyOiAke2R9IgogIHByaW50ZiAnJXMnICIkZCIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbG9jayAtLS0tCkNTX0xPQ0tfUEFUSD0nJwoKY3NfbG9jaygpIHsKICBsb2NhbCBuYW1lPSR7MTotbG9ja30gc2QgZgogIHNkPSIkKGNzX3N0YXRlX2RpcikiCiAgZj0iJHtzZH0vJHtuYW1lfS5sb2NrIgogIGlmIGNvbW1hbmQgLXYgZmxvY2sgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBleGVjIDk+IiRmIiB8fCBkaWUgImNhbm5vdCBvcGVuIGxvY2s6ICR7Zn0iCiAgICBmbG9jayAtbiA5IHx8IGRpZSAiYW5vdGhlciBpbnN0YW5jZSBpcyBhbHJlYWR5IHJ1bm5pbmcgKCR7Zn0pIgogICAgQ1NfTE9DS19QQVRIPSIkZiIKICBlbHNlCiAgICBsb2NhbCBkPSIke2Z9LmQiCiAgICBta2RpciAtLSAiJGQiIDI+L2Rldi9udWxsIHx8IGRpZSAiYW5vdGhlciBpbnN0YW5jZSBpcyBhbHJlYWR5IHJ1bm5pbmcgKCR7ZH0pIgogICAgcHJpbnRmICclc1xuJyAiJCQiID4iJHtkfS9waWQiCiAgICBDU19MT0NLX1BBVEg9IiRkIgogIGZpCn0KCmNzX3VubG9jaygpIHsKICBbWyAtbiAiJHtDU19MT0NLX1BBVEg6LX0iIF1dIHx8IHJldHVybiAwCiAgaWYgW1sgLWQgIiRDU19MT0NLX1BBVEgiIF1dOyB0aGVuCiAgICBybSAtcmYgLS0gIiRDU19MT0NLX1BBVEgiCiAgZWxzZQogICAgOiA+IiRDU19MT0NLX1BBVEgiIDI+L2Rldi9udWxsIHx8IHRydWUKICBmaQogIENTX0xPQ0tfUEFUSD0nJwogIHJldHVybiAwCn0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcGlkZmlsZXMgLS0tLQpjc19yZWFkX3BpZCgpIHsgW1sgLXMgIiR7MTotfSIgXV0gJiYgY2F0IC0tICIkMSIgMj4vZGV2L251bGwgfHwgcHJpbnRmICcnOyB9CmNzX3dyaXRlX3BpZCgpIHsgcHJpbnRmICclc1xuJyAiJDIiID4iJDEiOyB9Cgpjc19hbGl2ZSgpIHsKICBsb2NhbCBwCiAgcD0iJChjc19yZWFkX3BpZCAiJHsxOi19IikiCiAgW1sgLW4gIiRwIiBdXSB8fCByZXR1cm4gMQogIGtpbGwgLTAgIiRwIiAyPi9kZXYvbnVsbAp9Cgpjc19zdG9wX3BpZGZpbGUoKSB7CiAgbG9jYWwgZj0kMSBsYWJlbD0kezI6LXByb2Nlc3N9IHAgaQogIHA9IiQoY3NfcmVhZF9waWQgIiRmIikiCiAgcm0gLWYgLS0gIiRmIgogIFtbIC1uICIkcCIgXV0gfHwgcmV0dXJuIDAKICBraWxsIC0wICIkcCIgMj4vZGV2L251bGwgfHwgcmV0dXJuIDAKICBsb2cgInN0b3BwaW5nICR7bGFiZWx9IChwaWQgJHtwfSkiCiAga2lsbCAtVEVSTSAiJHAiIDI+L2Rldi9udWxsIHx8IHRydWUKICBmb3IgaSBpbiB7MS4uMTV9OyBkbwogICAga2lsbCAtMCAiJHAiIDI+L2Rldi9udWxsIHx8IHJldHVybiAwCiAgICBzbGVlcCAwLjMKICBkb25lCiAga2lsbCAtS0lMTCAiJHAiIDI+L2Rldi9udWxsIHx8IHRydWUKICByZXR1cm4gMAp9CgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBodHRwIC0tLS0KY3Nfd2FpdF9odHRwKCkgewogIGxvY2FsIHVybD0kMSB0aW1lb3V0PSR7MjotNjB9IGkKICBmb3IgKChpID0gMDsgaSA8IHRpbWVvdXQ7IGkrKykpOyBkbwogICAgIyAtLW5vcHJveHk6IENvbGFiL0p1cHl0ZXIgc29tZXRpbWVzIGV4cG9ydHMgaHR0cF9wcm94eSwgd2hpY2ggd291bGQgc2VuZAogICAgIyBsb29wYmFjayBjaGVja3MgdGhyb3VnaCBhIHByb3h5IHRoYXQgY2Fubm90IHJlYWNoIDEyNy4wLjAuMS4KICAgIGlmIGN1cmwgLWZzUyAtLW1heC10aW1lIDIgLS1ub3Byb3h5ICcqJyAiJHVybCIgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICAgIHJldHVybiAwCiAgICBmaQogICAgc2xlZXAgMQogIGRvbmUKICByZXR1cm4gMQp9Cgpjc19wb3J0X2luX3VzZSgpIHsKICBsb2NhbCBwPSQxCiAgaWYgY29tbWFuZCAtdiBzcyA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIHNzIC1sdG5IIDI+L2Rldi9udWxsIHwgYXdrIC12IHBhdD0iOiR7cH1cJCIgJyQ0IH4gcGF0IHsgZm91bmQgPSAxIH0gRU5EIHsgZXhpdCAhZm91bmQgfScKICBlbHNlCiAgICAoZXhlYyAzPD4iL2Rldi90Y3AvMTI3LjAuMC4xLyR7cH0iKSAyPi9kZXYvbnVsbAogIGZpCn0KCiMgRXN0YWJsaXNoZWQgY29ubmVjdGlvbnMgb24gYSBsb2NhbCBwb3J0LiBVc2VkIGJ5IHRoZSBpZGxlIHdhdGNoZG9nOiBjb3VudGluZwojIHNvY2tldHMgaXMgY2hlYXAgYW5kIG5lZWRzIG5vdGhpbmcgZnJvbSB0aGUgZWRpdG9yIGl0c2VsZi4KY3NfYWN0aXZlX2Nvbm5zKCkgewogIGxvY2FsIHA9JDEgbgogIGlmIGNvbW1hbmQgLXYgc3MgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBuPSIkKHNzIC1IdG4gc3RhdGUgZXN0YWJsaXNoZWQgInNwb3J0ID0gOiR7cH0iIDI+L2Rldi9udWxsIHwgd2MgLWwgfCB0ciAtZCAnICcpIgogICAgcHJpbnRmICclcycgIiR7bjotMH0iCiAgICByZXR1cm4gMAogIGZpCiAgbj0iJChweXRob24zIC1jICcKaW1wb3J0IHN5cwpwb3J0ID0gaW50KHN5cy5hcmd2WzFdKQp3YW50ID0gIiUwNFgiICUgcG9ydApuID0gMApmb3IgZiBpbiAoIi9wcm9jL25ldC90Y3AiLCAiL3Byb2MvbmV0L3RjcDYiKToKICAgIHRyeToKICAgICAgICByb3dzID0gb3BlbihmKS5yZWFkKCkuc3BsaXRsaW5lcygpWzE6XQogICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgY29udGludWUKICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgYyA9IHIuc3BsaXQoKQogICAgICAgIGlmIGxlbihjKSA+IDMgYW5kIGNbMV0uc3BsaXQoIjoiKVsxXS51cHBlcigpID09IHdhbnQgYW5kIGNbM10gPT0gIjAxIjoKICAgICAgICAgICAgbiArPSAxCnByaW50KG4pCicgIiRwIiAyPi9kZXYvbnVsbCB8fCB0cnVlKSIKICBwcmludGYgJyVzJyAiJHtuOi0wfSIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBwYXNzd29yZCAtLS0tCiMgUGVyc2lzdGVkIGluIHRoZSBzdGF0ZSBkaXIgc28gYSByZXN0YXJ0IGRvZXMgbm90IGludmFsaWRhdGUgYW4gb3BlbiB0YWIuCiMgQ1NfUEFTU1dPUkQgb3ZlcnJpZGVzIHdpdGhvdXQgdG91Y2hpbmcgZGlzay4KY3NfcGFzc3dvcmQoKSB7CiAgbG9jYWwgc2Q9JDEgZiBwCiAgZj0iJHtzZH0vcGFzc3dvcmQiCiAgaWYgW1sgLW4gIiR7Q1NfUEFTU1dPUkQ6LX0iIF1dOyB0aGVuCiAgICBwcmludGYgJyVzJyAiJENTX1BBU1NXT1JEIgogICAgcmV0dXJuIDAKICBmaQogIGlmIFtbIC1zICIkZiIgXV07IHRoZW4KICAgIGNhdCAtLSAiJGYiCiAgICByZXR1cm4gMAogIGZpCiAgcD0iJChMQ19BTEw9QyB0ciAtZGMgJ0EtWmEtejAtOScgPC9kZXYvdXJhbmRvbSAyPi9kZXYvbnVsbCB8IGhlYWQgLWMgMjQgfHwgdHJ1ZSkiCiAgW1sgJHsjcH0gLWVxIDI0IF1dIHx8IGRpZSAiZmFpbGVkIHRvIGdlbmVyYXRlIGEgcGFzc3dvcmQiCiAgKCB1bWFzayAwNzc7IHByaW50ZiAnJXMnICIkcCIgPiIkZiIgKQogIGNobW9kIDYwMCAtLSAiJGYiIDI+L2Rldi9udWxsIHx8IHRydWUKICBwcmludGYgJyVzJyAiJHAiCn0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGxvZ3MgLS0tLQpjc19yb3RhdGVfbG9nKCkgewogIGxvY2FsIGY9JDEgbWF4PSR7MjotNTI0Mjg4MH0gc3oKICBbWyAtZiAiJGYiIF1dIHx8IHJldHVybiAwCiAgc3o9IiQod2MgLWMgPCIkZiIgMj4vZGV2L251bGwgfHwgcHJpbnRmIDApIgogIGlmICgoc3ogPiBtYXgpKTsgdGhlbgogICAgbXYgLWYgLS0gIiRmIiAiJHtmfS4xIgogIGZpCiAgcmV0dXJuIDAKfQoKY3NfZHVtcF9sb2coKSB7CiAgbG9jYWwgZj0kMSBuPSR7MjotNDB9CiAgW1sgLXIgIiRmIiBdXSB8fCByZXR1cm4gMAogIHdhcm4gImxhc3QgJHtufSBsaW5lcyBvZiAke2Z9OiIKICB0YWlsIC1uICIkbiIgLS0gIiRmIiA+JjIgfHwgdHJ1ZQp9Cg=='
    ),
    'install.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvaW5zdGFsbC5zaCDigJQgaW5zdGFsbCBjb2RlLXNlcnZlciArIGEgdHVubmVsIGFnZW50IGluc2lkZSBhIENvbGFiIFZNLgojCiMgICBiYXNoIGNvbGFiL2luc3RhbGwuc2gKIyAgIFdJVEhfQ0xPVURGTEFSRUQ9MSBiYXNoIGNvbGFiL2luc3RhbGwuc2gKIyAgIENTX1ZFUlNJT049NC4xNDUuMiBiYXNoIGNvbGFiL2luc3RhbGwuc2ggICAgICMgcGluCiMgICBDU19DQUNIRV9ESVI9L2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8uY3MtY2FjaGUgYmFzaCBjb2xhYi9pbnN0YWxsLnNoCiMKIyBUaGUgcmVsZWFzZSB0YXJiYWxsIGlzIGNhY2hlZCBpbiB0aGUgc3RhdGUgZGlyLiBQb2ludCBDU19DQUNIRV9ESVIgYXQgRHJpdmUKIyBhbmQgYSBmcmVzaCBydW50aW1lIHJlaW5zdGFsbHMgaW4gc2Vjb25kcyBpbnN0ZWFkIG9mIHJlLWRvd25sb2FkaW5nIH4xMDAgTUIuCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKQ1NfUFJFRklYPSIke0NTX1BSRUZJWDotL3Vzci9sb2NhbH0iCkNTX1ZFUlNJT049IiR7Q1NfVkVSU0lPTjotfSIKV0lUSF9DTE9VREZMQVJFRD0iJHtXSVRIX0NMT1VERkxBUkVEOi0wfSIKU0tJUF9BUFQ9IiR7U0tJUF9BUFQ6LTB9IgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSByZXNvbHZlIHRoZSB2ZXJzaW9uIC0tCnJlc29sdmVfdmVyc2lvbigpIHsKICBpZiBbWyAtbiAiJENTX1ZFUlNJT04iIF1dOyB0aGVuCiAgICBwcmludGYgJyVzJyAiJHtDU19WRVJTSU9OI3Z9IgogICAgcmV0dXJuIDAKICBmaQogIGxvY2FsIGluc3RhbGxlZAogIGluc3RhbGxlZD0iJChjb2RlLXNlcnZlciAtLXZlcnNpb24gMj4vZGV2L251bGwgfCBhd2sgJ05SPT0xIHsgcHJpbnQgJDEgfScgfHwgdHJ1ZSkiCiAgaWYgW1sgLW4gIiRpbnN0YWxsZWQiIF1dOyB0aGVuCiAgICBwcmludGYgJyVzJyAiJGluc3RhbGxlZCIKICAgIHJldHVybiAwCiAgZmkKICAjIEhFQUQgb24gL3JlbGVhc2VzL2xhdGVzdCBmb2xsb3dzIHRoZSByZWRpcmVjdCB0byAvcmVsZWFzZXMvdGFnL3ZYLlkuWiBhbmQKICAjIGF2b2lkcyB0aGUgdW5hdXRoZW50aWNhdGVkIEdpdEh1YiBBUEkgcmF0ZSBsaW1pdCB0aGF0IHNoYXJlZCBDb2xhYiBlZ3Jlc3MKICAjIElQcyBoaXQgY29uc3RhbnRseS4KICBsb2NhbCB0YWcKICB0YWc9IiQoY3VybCAtZnNTTCAtbyAvZGV2L251bGwgLXcgJyV7dXJsX2VmZmVjdGl2ZX0nIFwKICAgIGh0dHBzOi8vZ2l0aHViLmNvbS9jb2Rlci9jb2RlLXNlcnZlci9yZWxlYXNlcy9sYXRlc3QgMj4vZGV2L251bGwgfHwgdHJ1ZSkiCiAgdGFnPSIke3RhZyMjKi90YWcvdn0iCiAgaWYgW1sgISAiJHRhZyIgPX4gXlswLTldK1wuWzAtOV0rXC5bMC05XSskIF1dOyB0aGVuCiAgICBkaWUgImNvdWxkIG5vdCByZXNvbHZlIHRoZSBsYXRlc3QgY29kZS1zZXJ2ZXIgdmVyc2lvbjsgc2V0IENTX1ZFUlNJT04iCiAgZmkKICBwcmludGYgJyVzJyAiJHRhZyIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcGFja2FnZXMgLS0KaW5zdGFsbF9wYWNrYWdlcygpIHsKICBsb2NhbCBwa2dzCiAgcGtncz0iY2EtY2VydGlmaWNhdGVzIGN1cmwgd2dldCB4ei11dGlscyBsaWJuc3MzIGxpYnNlY3JldC0xLTAgbGlieGtiZmlsZTEgbGliZ2JtMSBsaWJ4c2htZmVuY2UxICQoY3NfYXNvdW5kX3BrZykiCiAgbG9nICJpbnN0YWxsaW5nIHJ1bnRpbWUgbGlicmFyaWVzOiAke3BrZ3N9IgogIGNzX3Jvb3QgYXB0LWdldCB1cGRhdGUgLXFxCiAgIyBzaGVsbGNoZWNrIGRpc2FibGU9U0MyMDg2ICAjIGRlbGliZXJhdGUgd29yZCBzcGxpdHRpbmcgb2YgdGhlIHBhY2thZ2UgbGlzdAogIERFQklBTl9GUk9OVEVORD1ub25pbnRlcmFjdGl2ZSBjc19yb290IGFwdC1nZXQgaW5zdGFsbCAteSAtcXEgLS1uby1pbnN0YWxsLXJlY29tbWVuZHMgJHBrZ3MKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29kZS1zZXJ2ZXIgLS0KIyBPZmZpY2lhbCBzdGFuZGFsb25lIGxheW91dCwgcmVwbGljYXRlZCBzbyBgY29kZS1zZXJ2ZXJgIHJlc29sdmVzIG5vcm1hbGx5OgojICAgL3Vzci9sb2NhbC9saWIvY29kZS1zZXJ2ZXIgLT4gY29kZS1zZXJ2ZXItPHZlcj4KIyAgIC91c3IvbG9jYWwvYmluL2NvZGUtc2VydmVyIC0+IC4uL2xpYi9jb2RlLXNlcnZlci9iaW4vY29kZS1zZXJ2ZXIKaW5zdGFsbF9jb2RlX3NlcnZlcigpIHsKICBsb2NhbCB2ZXJzaW9uPSQxIGFyY2ggY2FjaGUgdGFyYmFsbCByb290IGxpbmsKCiAgYXJjaD0iJChjc19hcmNoKSIKICBjYWNoZT0iJChjc19jYWNoZV9kaXIpIgogIHRhcmJhbGw9IiR7Y2FjaGV9L2NvZGUtc2VydmVyLSR7dmVyc2lvbn0tbGludXgtJHthcmNofS50YXIuZ3oiCiAgcm9vdD0iJHtDU19QUkVGSVh9L2xpYi9jb2RlLXNlcnZlci0ke3ZlcnNpb259IgogIGxpbms9IiR7Q1NfUFJFRklYfS9saWIvY29kZS1zZXJ2ZXIiCgogIGlmIFtbIC14ICIke3Jvb3R9L2Jpbi9jb2RlLXNlcnZlciIgXV07IHRoZW4KICAgIG9rICJjb2RlLXNlcnZlciAke3ZlcnNpb259IGFscmVhZHkgdW5wYWNrZWQgYXQgJHtyb290fSIKICBlbHNlCiAgICBpZiBbWyAhIC1zICIkdGFyYmFsbCIgXV07IHRoZW4KICAgICAgbG9jYWwgdXJsPSJodHRwczovL2dpdGh1Yi5jb20vY29kZXIvY29kZS1zZXJ2ZXIvcmVsZWFzZXMvZG93bmxvYWQvdiR7dmVyc2lvbn0vY29kZS1zZXJ2ZXItJHt2ZXJzaW9ufS1saW51eC0ke2FyY2h9LnRhci5neiIKICAgICAgbG9nICJkb3dubG9hZGluZyAke3ZlcnNpb259ICgke2FyY2h9KSIKICAgICAgaWYgISBjdXJsIC1mTCAtLXJldHJ5IDMgLS1yZXRyeS1kZWxheSAyIC0jIC1vICIke3RhcmJhbGx9LnBhcnQiICIkdXJsIjsgdGhlbgogICAgICAgIHJtIC1mIC0tICIke3RhcmJhbGx9LnBhcnQiCiAgICAgICAgd2FybiAidGFyYmFsbCBkb3dubG9hZCBmYWlsZWQ7IGZhbGxpbmcgYmFjayB0byB0aGUgb2ZmaWNpYWwgaW5zdGFsbCBzY3JpcHQiCiAgICAgICAgY3VybCAtZnNTTCBodHRwczovL2NvZGUtc2VydmVyLmRldi9pbnN0YWxsLnNoIFwKICAgICAgICAgIHwgc2ggLXMgLS0gLS1tZXRob2Q9c3RhbmRhbG9uZSAtLXByZWZpeD0iJENTX1BSRUZJWCIgLS12ZXJzaW9uPSIkdmVyc2lvbiIKICAgICAgICByZXR1cm4gMAogICAgICBmaQogICAgICBtdiAtZiAtLSAiJHt0YXJiYWxsfS5wYXJ0IiAiJHRhcmJhbGwiCiAgICBlbHNlCiAgICAgIG9rICJ1c2luZyBjYWNoZWQgdGFyYmFsbCAkKGJhc2VuYW1lIC0tICIkdGFyYmFsbCIpIgogICAgZmkKCiAgICBsb2cgInVucGFja2luZyBpbnRvICR7Q1NfUFJFRklYfS9saWIiCiAgICBsb2NhbCBzdGFnZSBleHRyYWN0ZWQKICAgIHN0YWdlPSIkKG1rdGVtcCAtZCkiCiAgICB0YXIgLXh6ZiAiJHRhcmJhbGwiIC1DICIkc3RhZ2UiCiAgICBleHRyYWN0ZWQ9IiQoZmluZCAiJHN0YWdlIiAtbWF4ZGVwdGggMSAtbWluZGVwdGggMSAtdHlwZSBkIHwgaGVhZCAtMSkiCiAgICBpZiBbWyAteiAiJGV4dHJhY3RlZCIgXV07IHRoZW4KICAgICAgcm0gLXJmIC0tICIkc3RhZ2UiCiAgICAgIGRpZSAidW5leHBlY3RlZCB0YXJiYWxsIGxheW91dCBpbiAkKGJhc2VuYW1lIC0tICIkdGFyYmFsbCIpIgogICAgZmkKICAgIGNzX3Jvb3Qgcm0gLXJmIC0tICIkcm9vdCIKICAgIGNzX3Jvb3QgbXYgLS0gIiRleHRyYWN0ZWQiICIkcm9vdCIKICAgIHJtIC1yZiAtLSAiJHN0YWdlIgogIGZpCgogIGNzX3Jvb3QgbG4gLXNmbiAtLSAiY29kZS1zZXJ2ZXItJHt2ZXJzaW9ufSIgIiRsaW5rIgogIGNzX3Jvb3QgbWtkaXIgLXAgLS0gIiR7Q1NfUFJFRklYfS9iaW4iCiAgY3Nfcm9vdCBsbiAtc2ZuIC0tICIuLi9saWIvY29kZS1zZXJ2ZXIvYmluL2NvZGUtc2VydmVyIiAiJHtDU19QUkVGSVh9L2Jpbi9jb2RlLXNlcnZlciIKICBoYXNoIC1yCgogIFtbIC14ICIke0NTX1BSRUZJWH0vYmluL2NvZGUtc2VydmVyIiBdXSB8fCBkaWUgImNvZGUtc2VydmVyIG5vdCBvbiBQQVRIIGF0ICR7Q1NfUFJFRklYfS9iaW4iCiAgb2sgImNvZGUtc2VydmVyICQoY29kZS1zZXJ2ZXIgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCn0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG5ncm9rIC0tCmluc3RhbGxfbmdyb2soKSB7CiAgaWYgY29tbWFuZCAtdiBuZ3JvayA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIG9rICJuZ3JvayAkKG5ncm9rIC0tdmVyc2lvbiAyPi9kZXYvbnVsbCB8IGhlYWQgLTEpIgogICAgcmV0dXJuIDAKICBmaQogIGxvY2FsIGFyY2ggdG1wCiAgYXJjaD0iJChjc19hcmNoKSIKICB0bXA9IiQobWt0ZW1wIC1kKSIKICBsb2cgImluc3RhbGxpbmcgbmdyb2sgYWdlbnQgKCR7YXJjaH0pIgogIGN1cmwgLWZzU0wgLS1yZXRyeSAzIFwKICAgICJodHRwczovL2Jpbi5lcXVpbm94LmlvL2MvYk55ajFtUVZZNGMvbmdyb2stdjMtc3RhYmxlLWxpbnV4LSR7YXJjaH0udGd6IiBcCiAgICAtbyAiJHt0bXB9L25ncm9rLnRneiIgfHwgZGllICJuZ3JvayBkb3dubG9hZCBmYWlsZWQiCiAgdGFyIC14emYgIiR7dG1wfS9uZ3Jvay50Z3oiIC1DICIkdG1wIgogIGNzX3Jvb3QgaW5zdGFsbCAtbSAwNzU1ICIke3RtcH0vbmdyb2siICIke0NTX1BSRUZJWH0vYmluL25ncm9rIgogIHJtIC1yZiAtLSAiJHRtcCIKICBoYXNoIC1yCiAgb2sgIm5ncm9rICQobmdyb2sgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCn0KCmNvbmZpZ3VyZV9uZ3JvaygpIHsKICBpZiBbWyAtbiAiJHtOR1JPS19UT0tFTjotfSIgXV07IHRoZW4KICAgIGxvZyAiY29uZmlndXJpbmcgbmdyb2sgYXV0aHRva2VuIgogICAgbmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gIiROR1JPS19UT0tFTiIgPi9kZXYvbnVsbCAyPiYxIFwKICAgICAgfHwgd2FybiAibmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gZmFpbGVkIgogICAgcmV0dXJuIDAKICBmaQogIGlmIG5ncm9rIGNvbmZpZyBjaGVjayA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIG9rICJuZ3JvayBhdXRodG9rZW4gYWxyZWFkeSBjb25maWd1cmVkIgogICAgcmV0dXJuIDAKICBmaQogIHdhcm4gIm5vIE5HUk9LX1RPS0VOIGFuZCBubyBhdXRodG9rZW4gY29uZmlndXJlZCDigJQgbmdyb2sgd2lsbCBmYWlsIHVudGlsIHlvdSBydW46IgogIHdhcm4gIiAgbmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gPHRva2VuPiAgICAjIGRhc2hib2FyZC5uZ3Jvay5jb20vZ2V0LXN0YXJ0ZWQveW91ci1hdXRodG9rZW4iCiAgcmV0dXJuIDAKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2xvdWRmbGFyZWQgLS0KaW5zdGFsbF9jbG91ZGZsYXJlZCgpIHsKICBpZiBjb21tYW5kIC12IGNsb3VkZmxhcmVkID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgb2sgImNsb3VkZmxhcmVkICQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKICAgIHJldHVybiAwCiAgZmkKICBsb2NhbCBhcmNoCiAgYXJjaD0iJChjc19hcmNoKSIKICBsb2cgImluc3RhbGxpbmcgY2xvdWRmbGFyZWQgKCR7YXJjaH0pIgogIGN1cmwgLWZzU0wgLS1yZXRyeSAzIFwKICAgICJodHRwczovL2dpdGh1Yi5jb20vY2xvdWRmbGFyZS9jbG91ZGZsYXJlZC9yZWxlYXNlcy9sYXRlc3QvZG93bmxvYWQvY2xvdWRmbGFyZWQtbGludXgtJHthcmNofSIgXAogICAgLW8gL3RtcC9jbG91ZGZsYXJlZC5iaW4gfHwgZGllICJjbG91ZGZsYXJlZCBkb3dubG9hZCBmYWlsZWQiCiAgY3Nfcm9vdCBpbnN0YWxsIC1tIDA3NTUgL3RtcC9jbG91ZGZsYXJlZC5iaW4gIiR7Q1NfUFJFRklYfS9iaW4vY2xvdWRmbGFyZWQiCiAgcm0gLWYgLS0gL3RtcC9jbG91ZGZsYXJlZC5iaW4KICBoYXNoIC1yCiAgb2sgImNsb3VkZmxhcmVkICQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1haW4gLS0KbWFpbigpIHsKICBsb2NhbCBzZAogIHNkPSIkKGNzX3N0YXRlX2RpcikiCgogIGxvY2FsIHZlcnNpb24KICB2ZXJzaW9uPSIkKHJlc29sdmVfdmVyc2lvbikiCiAgbG9nICJ0YXJnZXQgY29kZS1zZXJ2ZXIgdmVyc2lvbjogJHt2ZXJzaW9ufSIKCiAgaWYgW1sgIiRTS0lQX0FQVCIgIT0gIjEiIF1dOyB0aGVuCiAgICBpZiBjb21tYW5kIC12IGNvZGUtc2VydmVyID4vZGV2L251bGwgMj4mMSAmJiBbWyAiJChjb2RlLXNlcnZlciAtLXZlcnNpb24gMj4vZGV2L251bGwgfCBhd2sgJ05SPT0xIHsgcHJpbnQgJDEgfScpIiA9PSAiJHZlcnNpb24iIF1dOyB0aGVuCiAgICAgIG9rICJydW50aW1lIGxpYnJhcmllcyBhc3N1bWVkIHByZXNlbnQgKGNvZGUtc2VydmVyICR7dmVyc2lvbn0gYWxyZWFkeSBpbnN0YWxsZWQpIgogICAgZWxzZQogICAgICBpbnN0YWxsX3BhY2thZ2VzCiAgICBmaQogIGZpCgogIGluc3RhbGxfY29kZV9zZXJ2ZXIgIiR2ZXJzaW9uIgogIGluc3RhbGxfbmdyb2sKICBjb25maWd1cmVfbmdyb2sKICBpZiBbWyAiJFdJVEhfQ0xPVURGTEFSRUQiID09ICIxIiBdXTsgdGhlbgogICAgaW5zdGFsbF9jbG91ZGZsYXJlZAogIGZpCgogIHByaW50ZiAnJXNcbicgIiR2ZXJzaW9uIiA+IiR7c2R9Ly5pbnN0YWxsZWQtdmVyc2lvbiIKCiAgY2F0IDw8RU9GCgogIGluc3RhbGwgY29tcGxldGUKCiAgbmV4dCAgICAgYmFzaCBjb2xhYi9zZXJ2ZS5zaAogIG9yICAgICAgIGJhc2ggY29sYWIvc3VwZXJ2aXNlLnNoICAgICAgKHJlc3RhcnRzICsgaWRsZS9tYXgtcnVudGltZSB3YXRjaGRvZykKICBjaGVjayAgICBiYXNoIGNvbGFiL2RvY3Rvci5zaAoKRU9GCn0KCm1haW4gIiRAIgo='
    ),
    'serve.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc2VydmUuc2gg4oCUIHN0YXJ0IGNvZGUtc2VydmVyIGFuZCBleHBvc2UgaXQgdGhyb3VnaCBhIHR1bm5lbC4KIwojICAgYmFzaCBjb2xhYi9zZXJ2ZS5zaCAgICAgICAgICAgICAgICAgICAgICAjIG5ncm9rLCBiYWNrZ3JvdW5kLCBwcmludHMgVVJMCiMgICBiYXNoIGNvbGFiL3NlcnZlLnNoIC0tZm9yZWdyb3VuZCAgICAgICAgICMgYmxvY2sgKHVzZWQgYnkgc3VwZXJ2aXNlLnNoKQojICAgQ1NfVFVOTkVMPWNsb3VkZmxhcmVkIGJhc2ggY29sYWIvc2VydmUuc2gKIyAgIE5HUk9LX0RPTUFJTj1teS1kZXYubmdyb2stZnJlZS5hcHAgYmFzaCBjb2xhYi9zZXJ2ZS5zaCAgICMgc3RhYmxlIGhvc3RuYW1lCiMKIyBOb3RoaW5nIGhlcmUgbW9kaWZpZXMgY29kZS1zZXJ2ZXIuIEl0IGlzIGEgcGxhaW4gTm9kZSBIVFRQIHNlcnZlcjsgdGhpcyBzY3JpcHQKIyBvbmx5IHBpY2tzIGZsYWdzIHRoYXQgc3VpdCBhbiBlcGhlbWVyYWwgQ29sYWIgVk0uCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKRk9SRUdST1VORD0wClRVTk5FTD0iJHtDU19UVU5ORUw6LWF1dG99IgpQT1JUPSIke0NTX1BPUlQ6LTgwODB9IgpBVVRIPSJwYXNzd29yZCIKCnVzYWdlKCkgewogIGNhdCA8PCdFT0YnCnVzYWdlOiBzZXJ2ZS5zaCBbb3B0aW9uc10KCiAgLS1mb3JlZ3JvdW5kICAgICByZW1haW4gaW4gdGhlIGZvcmVncm91bmQgdW50aWwgYSBjaGlsZCBleGl0cwogIC0tdHVubmVsPU5BTUUgICAgYXV0byB8IG5ncm9rIHwgY2xvdWRmbGFyZWQgfCBub25lIChkZWZhdWx0OiAkQ1NfVFVOTkVMLCBhdXRvKQogIC0tcG9ydD1OICAgICAgICAgbG9jYWwgcG9ydCAgICAgICAgICAgICAgICAgICAgICAgIChkZWZhdWx0OiAkQ1NfUE9SVCwgODA4MCkKICAtLW5vLWF1dGggICAgICAgIGRpc2FibGUgYXV0aGVudGljYXRpb24gKHJlcXVpcmVzIENTX0FMTE9XX0lOU0VDVVJFPTEpCiAgLWgsIC0taGVscApFT0YKfQoKd2hpbGUgKCgkIykpOyBkbwogIGNhc2UgIiQxIiBpbgogICAgLS1mb3JlZ3JvdW5kKSBGT1JFR1JPVU5EPTEgOzsKICAgIC0tdHVubmVsPSopICAgVFVOTkVMPSIkezEjKj19IiA7OwogICAgLS1wb3J0PSopICAgICBQT1JUPSIkezEjKj19IiA7OwogICAgLS1uby1hdXRoKSAgICBBVVRIPSJub25lIiA7OwogICAgLWh8LS1oZWxwKSAgICB1c2FnZTsgZXhpdCAwIDs7CiAgICAqKSAgICAgICAgICAgIGRpZSAidW5rbm93biBhcmd1bWVudDogJDEgKHRyeSAtLWhlbHApIiA7OwogIGVzYWMKICBzaGlmdApkb25lCgpbWyAiJFBPUlQiID1+IF5bMC05XSskIF1dIHx8IGRpZSAiaW52YWxpZCBwb3J0OiAke1BPUlR9IgpjYXNlICIkVFVOTkVMIiBpbgogIGF1dG98bmdyb2t8Y2xvdWRmbGFyZWR8bm9uZSkgOzsKICAqKSBkaWUgIi0tdHVubmVsIG11c3QgYmUgYXV0bywgbmdyb2ssIGNsb3VkZmxhcmVkIG9yIG5vbmUgKGdvdDogJHtUVU5ORUx9KSIgOzsKZXNhYwoKIyBhdXRvOiBuZ3JvayBpZiBpdCBoYXMgY3JlZGVudGlhbHMsIG90aGVyd2lzZSBjbG91ZGZsYXJlZCBxdWljayB0dW5uZWwsIHdoaWNoCiMgbmVlZHMgbm8gYWNjb3VudCBhdCBhbGwuIFRoYXQgbWFrZXMgYSBiYXJlIG5vdGVib29rIHJ1biB3b3JrIHdpdGggemVybyBzZXR1cC4KaWYgW1sgIiRUVU5ORUwiID09ICJhdXRvIiBdXTsgdGhlbgogIGlmIFtbIC1uICIke05HUk9LX1RPS0VOOi19IiBdXTsgdGhlbgogICAgVFVOTkVMPW5ncm9rCiAgZWxpZiBjb21tYW5kIC12IG5ncm9rID4vZGV2L251bGwgMj4mMSAmJiBuZ3JvayBjb25maWcgY2hlY2sgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBUVU5ORUw9bmdyb2sKICBlbGlmIGNvbW1hbmQgLXYgY2xvdWRmbGFyZWQgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBUVU5ORUw9Y2xvdWRmbGFyZWQKICBlbHNlCiAgICBUVU5ORUw9bmdyb2sKICBmaQogIGxvZyAiYXV0by1zZWxlY3RlZCB0dW5uZWw6ICR7VFVOTkVMfSIKZmkKCmlmIFtbICIkQVVUSCIgPT0gIm5vbmUiIF1dOyB0aGVuCiAgW1sgIiR7Q1NfQUxMT1dfSU5TRUNVUkU6LTB9IiA9PSAiMSIgXV0gXAogICAgfHwgZGllICItLW5vLWF1dGggcmVxdWlyZXMgQ1NfQUxMT1dfSU5TRUNVUkU9MSIKICBbWyAiJFRVTk5FTCIgPT0gIm5vbmUiIF1dIFwKICAgIHx8IGRpZSAiLS1uby1hdXRoIGNhbm5vdCBiZSBjb21iaW5lZCB3aXRoIGEgcHVibGljIHR1bm5lbCIKZmkKCmNzX25lZWQgY29kZS1zZXJ2ZXIgInJ1bjogYmFzaCBjb2xhYi9pbnN0YWxsLnNoIgpjc19uZWVkIGN1cmwKY3NfbmVlZCBweXRob24zCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpjc19sb2NrIHNlcnZlCgpDU19QSUQ9JycKVFVOX1BJRD0nJwpQVUJMSUNfVVJMPScnCgpjbGVhbnVwX2NoaWxkcmVuKCkgewogIFtbIC1uICIkQ1NfUElEIiBdXSAmJiBjc19zdG9wX3BpZGZpbGUgIiR7U0R9L2NzLnBpZCIgY29kZS1zZXJ2ZXIKICBbWyAtbiAiJFRVTl9QSUQiIF1dICYmIGNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vJHtUVU5ORUx9LnBpZCIgIiRUVU5ORUwiCiAgcmV0dXJuIDAKfQoKb25fZXhpdCgpIHsKICBpZiAoKEZPUkVHUk9VTkQpKTsgdGhlbiBjbGVhbnVwX2NoaWxkcmVuOyBmaQogIGNzX3VubG9jawp9CnRyYXAgb25fZXhpdCBFWElUCiMgVHVybiBzaWduYWxzIGludG8gYSBub3JtYWwgZXhpdCBzbyB0aGUgRVhJVCB0cmFwIHRlYXJzIHRoZSBjaGlsZHJlbiBkb3duLgp0cmFwICdleGl0IDE0MycgVEVSTSBJTlQgSFVQCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RhbGUgaW5zdGFuY2VzIC0tLS0KbG9nICJjbGVhcmluZyBzdGFsZSBpbnN0YW5jZXMiCmNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vY3MucGlkIiBjb2RlLXNlcnZlcgpjc19zdG9wX3BpZGZpbGUgIiR7U0R9L25ncm9rLnBpZCIgbmdyb2sKY3Nfc3RvcF9waWRmaWxlICIke1NEfS9jbG91ZGZsYXJlZC5waWQiIGNsb3VkZmxhcmVkCgppZiBjc19wb3J0X2luX3VzZSAiJFBPUlQiOyB0aGVuCiAgZGllICJwb3J0ICR7UE9SVH0gaXMgc3RpbGwgaW4gdXNlIOKAlCBhbm90aGVyIHByb2Nlc3Mgb3ducyBpdCAodHJ5IC0tcG9ydD08bj4pIgpmaQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZXh0ZW5zaW9ucyAtLS0tCiMgSW5zdGFsbGVkIGJlZm9yZSB0aGUgc2VydmVyIHN0YXJ0cyBzbyB0aGUgbWFya2VyIGZpbGUgYW5kIHRoZSBleHRlbnNpb24gZGlyCiMgY2Fubm90IHJhY2Ugd2l0aCB0aGUgZXh0ZW5zaW9uIGhvc3QuCmluc3RhbGxfZXh0ZW5zaW9ucygpIHsKICBsb2NhbCBzZD0kMSBtYXJrZXIgZXh0CiAgbWFya2VyPSIke3NkfS8uZXh0ZW5zaW9ucy0kKHByaW50ZiAnJXMnICIkQ1NfRVhURU5TSU9OUyIgfCBja3N1bSB8IGN1dCAtZCcgJyAtZjEpIgogIGlmIFtbIC1mICIkbWFya2VyIiBdXTsgdGhlbgogICAgb2sgImV4dGVuc2lvbnMgYWxyZWFkeSBpbnN0YWxsZWQiCiAgICByZXR1cm4gMAogIGZpCiAgbG9jYWwgSUZTPScsJwogIGZvciBleHQgaW4gJENTX0VYVEVOU0lPTlM7IGRvCiAgICBbWyAtbiAiJGV4dCIgXV0gfHwgY29udGludWUKICAgIGxvZyAiaW5zdGFsbGluZyBleHRlbnNpb24gJHtleHR9IgogICAgaWYgY29kZS1zZXJ2ZXIgXAogICAgICAgIC0tdXNlci1kYXRhLWRpciAiJHtzZH0vZGF0YSIgXAogICAgICAgIC0tZXh0ZW5zaW9ucy1kaXIgIiR7c2R9L2V4dGVuc2lvbnMiIFwKICAgICAgICAtLWluc3RhbGwtZXh0ZW5zaW9uICIkZXh0IiAtLWZvcmNlIFwKICAgICAgICA+PiIke3NkfS9leHRlbnNpb25zLmxvZyIgMj4mMTsgdGhlbgogICAgICBvayAiZXh0ZW5zaW9uICR7ZXh0fSIKICAgIGVsc2UKICAgICAgd2FybiAiZXh0ZW5zaW9uICR7ZXh0fSBmYWlsZWQgKHNlZSAke3NkfS9leHRlbnNpb25zLmxvZykiCiAgICBmaQogIGRvbmUKICA6ID4iJG1hcmtlciIKfQppZiBbWyAtbiAiJHtDU19FWFRFTlNJT05TOi19IiBdXTsgdGhlbgogIGluc3RhbGxfZXh0ZW5zaW9ucyAiJFNEIgpmaQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjb2RlLXNlcnZlciAtLS0tCkNPREVfUEFTU1dPUkQ9IiQoY3NfcGFzc3dvcmQgIiRTRCIpIgoKY3NfYXJncz0oCiAgLS1iaW5kLWFkZHIgIjEyNy4wLjAuMToke1BPUlR9IgogIC0tYXV0aCAiJEFVVEgiCiAgLS11c2VyLWRhdGEtZGlyICIke1NEfS9kYXRhIgogIC0tZXh0ZW5zaW9ucy1kaXIgIiR7U0R9L2V4dGVuc2lvbnMiCiAgLS1kaXNhYmxlLXRlbGVtZXRyeQogIC0tZGlzYWJsZS11cGRhdGUtY2hlY2sKICAtLWRpc2FibGUtd29ya3NwYWNlLXRydXN0CikKCmxvZyAic3RhcnRpbmcgY29kZS1zZXJ2ZXIgb24gMTI3LjAuMC4xOiR7UE9SVH0iCmNzX3JvdGF0ZV9sb2cgIiR7U0R9L2NvZGUtc2VydmVyLmxvZyIKUEFTU1dPUkQ9IiRDT0RFX1BBU1NXT1JEIiBub2h1cCBjb2RlLXNlcnZlciAiJHtjc19hcmdzW0BdfSIgXAogID4iJHtTRH0vY29kZS1zZXJ2ZXIubG9nIiAyPiYxIDk+Ji0gJgpDU19QSUQ9JCEKY3Nfd3JpdGVfcGlkICIke1NEfS9jcy5waWQiICIkQ1NfUElEIgoKaWYgISBjc193YWl0X2h0dHAgImh0dHA6Ly8xMjcuMC4wLjE6JHtQT1JUfS9oZWFsdGh6IiAiJHtDU19IRUFMVEhfVElNRU9VVDotOTB9IjsgdGhlbgogIGNzX2R1bXBfbG9nICIke1NEfS9jb2RlLXNlcnZlci5sb2ciIDYwCiAgY3Nfc3RvcF9waWRmaWxlICIke1NEfS9jcy5waWQiIGNvZGUtc2VydmVyCiAgQ1NfUElEPScnCiAgZGllICJjb2RlLXNlcnZlciBkaWQgbm90IGJlY29tZSBoZWFsdGh5IgpmaQpvayAiY29kZS1zZXJ2ZXIgaGVhbHRoeSAocGlkICR7Q1NfUElEfSkiCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdHVubmVscyAtLS0tCm5ncm9rX3B1YmxpY191cmwoKSB7CiAgY3VybCAtZnNTIC0tbWF4LXRpbWUgMiAtLW5vcHJveHkgJyonICJodHRwOi8vMTI3LjAuMC4xOjQwNDAvYXBpL3R1bm5lbHMiIDI+L2Rldi9udWxsIFwKICAgIHwgcHl0aG9uMyAtYyAnCmltcG9ydCBqc29uLCBzeXMKdHJ5OgogICAgZGF0YSA9IGpzb24ubG9hZChzeXMuc3RkaW4pCmV4Y2VwdCBFeGNlcHRpb246CiAgICBzeXMuZXhpdCgwKQpmb3IgdCBpbiBkYXRhLmdldCgidHVubmVscyIsIFtdKToKICAgIHVybCA9IHQuZ2V0KCJwdWJsaWNfdXJsIiwgIiIpCiAgICBpZiB1cmwuc3RhcnRzd2l0aCgiaHR0cHM6Ly8iKToKICAgICAgICBwcmludCh1cmwpCiAgICAgICAgYnJlYWsKJyAyPi9kZXYvbnVsbCB8fCB0cnVlCn0KCnN0YXJ0X25ncm9rKCkgewogIGNzX25lZWQgbmdyb2sgInJ1bjogYmFzaCBjb2xhYi9pbnN0YWxsLnNoIgoKICBpZiBbWyAtbiAiJHtOR1JPS19UT0tFTjotfSIgXV07IHRoZW4KICAgIG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuICIkTkdST0tfVE9LRU4iID4vZGV2L251bGwgMj4mMSBcCiAgICAgIHx8IHdhcm4gIm5ncm9rIGF1dGh0b2tlbiByZWZyZXNoIGZhaWxlZCIKICBmaQoKICBsb2NhbCBhcmdzPShodHRwKQogIGlmIFtbIC1uICIke05HUk9LX0RPTUFJTjotfSIgXV07IHRoZW4KICAgIGFyZ3MrPSgiLS11cmw9aHR0cHM6Ly8ke05HUk9LX0RPTUFJTn0iKQogIGZpCiAgaWYgW1sgLW4gIiR7TkdST0tfQkFTSUNfQVVUSDotfSIgXV07IHRoZW4KICAgICMgRXh0cmEgZ2F0ZSBhdCBuZ3JvaydzIGVkZ2U6IHJlamVjdHMgc2Nhbm5lcnMgYmVmb3JlIHRoZXkgZXZlciByZWFjaAogICAgIyBjb2RlLXNlcnZlci4gRnJlZSB0aWVyIGhvc3RuYW1lcyBnZXQgcHJvYmVkIHdpdGhpbiBtaW51dGVzLgogICAgYXJncys9KCItLWJhc2ljLWF1dGg9JHtOR1JPS19CQVNJQ19BVVRIfSIpCiAgZmkKICBhcmdzKz0oIiRQT1JUIiAtLWxvZz1zdGRvdXQgLS1sb2ctZm9ybWF0PWpzb24pCgogIGxvZyAib3BlbmluZyBuZ3JvayB0dW5uZWwiCiAgY3Nfcm90YXRlX2xvZyAiJHtTRH0vbmdyb2subG9nIgogIG5vaHVwIG5ncm9rICIke2FyZ3NbQF19IiA+IiR7U0R9L25ncm9rLmxvZyIgMj4mMSA5PiYtICYKICBUVU5fUElEPSQhCiAgY3Nfd3JpdGVfcGlkICIke1NEfS9uZ3Jvay5waWQiICIkVFVOX1BJRCIKCiAgbG9jYWwgaQogIGZvciAoKGkgPSAwOyBpIDwgNDU7IGkrKykpOyBkbwogICAgUFVCTElDX1VSTD0iJChuZ3Jva19wdWJsaWNfdXJsKSIKICAgIFtbIC1uICIkUFVCTElDX1VSTCIgXV0gJiYgcmV0dXJuIDAKICAgIGtpbGwgLTAgIiRUVU5fUElEIiAyPi9kZXYvbnVsbCB8fCBicmVhawogICAgc2xlZXAgMQogIGRvbmUKCiAgY3NfZHVtcF9sb2cgIiR7U0R9L25ncm9rLmxvZyIgNDAKICB3YXJuICJubyBwdWJsaWMgVVJMIGZyb20gbmdyb2siCiAgd2FybiAiY29tbW9uIGNhdXNlczogbWlzc2luZyBhdXRodG9rZW4gKG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuIDx0PiksIgogIHdhcm4gImZyZWUtdGllciBzZXNzaW9uIGxpbWl0IChvbmUgYWdlbnQgYXQgYSB0aW1lKSwgb3IgYSByZXNlcnZlZC1kb21haW4gbWlzbWF0Y2giCiAgcmV0dXJuIDEKfQoKc3RhcnRfY2xvdWRmbGFyZWQoKSB7CiAgY3NfbmVlZCBjbG91ZGZsYXJlZCAicnVuOiBXSVRIX0NMT1VERkxBUkVEPTEgYmFzaCBjb2xhYi9pbnN0YWxsLnNoIgoKICBsb2NhbCBhcmdzPSh0dW5uZWwpCiAgaWYgW1sgLW4gIiR7Q0ZfVE9LRU46LX0iIF1dOyB0aGVuCiAgICAjIE5hbWVkIHR1bm5lbDogc3RhYmxlIGhvc3RuYW1lIG9uIGEgZG9tYWluIHlvdSBjb250cm9sLCBubyBpbnRlcnN0aXRpYWwKICAgICMgcGFnZSwgc3Vydml2ZXMgcmVzdGFydHMuIFRoaXMgaXMgdGhlIGFuc3dlciB0byAidGhlIG5ncm9rIFVSTCBrZWVwcwogICAgIyBjaGFuZ2luZyIg4oCUIHByb3BlciBob3N0aW5nIHJhdGhlciB0aGFuIGEgZGlzZ3Vpc2UuCiAgICBhcmdzKz0ocnVuIC0tdG9rZW4gIiRDRl9UT0tFTiIpCiAgZWxzZQogICAgYXJncys9KC0tdXJsICJodHRwOi8vMTI3LjAuMC4xOiR7UE9SVH0iKQogIGZpCiAgYXJncys9KC0tbm8tYXV0b3VwZGF0ZSkKCiAgbG9nICJvcGVuaW5nIGNsb3VkZmxhcmVkIHR1bm5lbCIKICBjc19yb3RhdGVfbG9nICIke1NEfS9jbG91ZGZsYXJlZC5sb2ciCiAgbm9odXAgY2xvdWRmbGFyZWQgIiR7YXJnc1tAXX0iID4iJHtTRH0vY2xvdWRmbGFyZWQubG9nIiAyPiYxIDk+Ji0gJgogIFRVTl9QSUQ9JCEKICBjc193cml0ZV9waWQgIiR7U0R9L2Nsb3VkZmxhcmVkLnBpZCIgIiRUVU5fUElEIgoKICBpZiBbWyAtbiAiJHtDRl9UVU5ORUxfVVJMOi19IiBdXTsgdGhlbgogICAgUFVCTElDX1VSTD0iJENGX1RVTk5FTF9VUkwiCiAgICByZXR1cm4gMAogIGZpCgogIGxvY2FsIGkKICBmb3IgKChpID0gMDsgaSA8IDQ1OyBpKyspKTsgZG8KICAgIFBVQkxJQ19VUkw9IiQoZ3JlcCAtb0UgJ2h0dHBzOi8vW2EtejAtOS1dK1wudHJ5Y2xvdWRmbGFyZVwuY29tJyBcCiAgICAgICIke1NEfS9jbG91ZGZsYXJlZC5sb2ciIDI+L2Rldi9udWxsIHwgaGVhZCAtMSB8fCB0cnVlKSIKICAgIFtbIC1uICIkUFVCTElDX1VSTCIgXV0gJiYgcmV0dXJuIDAKICAgIGtpbGwgLTAgIiRUVU5fUElEIiAyPi9kZXYvbnVsbCB8fCBicmVhawogICAgc2xlZXAgMQogIGRvbmUKCiAgY3NfZHVtcF9sb2cgIiR7U0R9L2Nsb3VkZmxhcmVkLmxvZyIgNDAKICB3YXJuICJubyBwdWJsaWMgVVJMIGZyb20gY2xvdWRmbGFyZWQiCiAgd2FybiAid2l0aCBDRl9UT0tFTiBzZXQsIHBhc3MgQ0ZfVFVOTkVMX1VSTD1odHRwczovL3lvdXIuaG9zdG5hbWUgc28gaXQgY2FuIGJlIHJlcG9ydGVkIgogIHJldHVybiAxCn0KCmNhc2UgIiRUVU5ORUwiIGluCiAgbmdyb2spICAgICAgIHN0YXJ0X25ncm9rICAgICAgIHx8IGRpZSAidHVubmVsIHN0YXJ0dXAgZmFpbGVkIiA7OwogIGNsb3VkZmxhcmVkKSBzdGFydF9jbG91ZGZsYXJlZCB8fCBkaWUgInR1bm5lbCBzdGFydHVwIGZhaWxlZCIgOzsKICBub25lKSAgICAgICAgUFVCTElDX1VSTD0iaHR0cDovLzEyNy4wLjAuMToke1BPUlR9IiA7Owplc2FjCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcmVwb3J0IC0tLS0KewogIHByaW50ZiAndXJsPSVzXG4nICIkUFVCTElDX1VSTCIKICBwcmludGYgJ3Bhc3N3b3JkPSVzXG4nICIkQ09ERV9QQVNTV09SRCIKICBwcmludGYgJ3R1bm5lbD0lc1xuJyAiJFRVTk5FTCIKICBwcmludGYgJ3BvcnQ9JXNcbicgIiRQT1JUIgogIHByaW50ZiAnc3RhdGU9JXNcbicgIiRTRCIKfSA+IiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIKY2htb2QgNjAwIC0tICIke1NEfS9jcmVkZW50aWFscy50eHQiIDI+L2Rldi9udWxsIHx8IHRydWUKCnByaW50ZiAnXG4nCmRpbSAgJyAg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAJwpwcmludGYgJyAgICVzICAlc1xuJyAiJChwcmludGYgJ1wwMzNbMW1jb2RlLXNlcnZlclwwMzNbMG0nKSIgIiRQVUJMSUNfVVJMIgpwcmludGYgJyAgIHBhc3N3b3JkICAgICAlc1xuJyAiJENPREVfUEFTU1dPUkQiCltbIC1uICIke05HUk9LX0JBU0lDX0FVVEg6LX0iIF1dICYmIHByaW50ZiAnICAgYmFzaWMgYXV0aCAgICVzXG4nICIkTkdST0tfQkFTSUNfQVVUSCIKZGltICAnICDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAnCnByaW50ZiAnXG4nCmRpbSAiICBzdGF0ZSAgICR7U0R9IgpkaW0gIiAgbG9ncyAgICAke1NEfS9jb2RlLXNlcnZlci5sb2cgICR7U0R9LyR7VFVOTkVMfS5sb2ciCmRpbSAiICBzdG9wICAgIGJhc2ggY29sYWIvc3RvcC5zaCIKcHJpbnRmICdcbicKCmlmIFtbICIkVFVOTkVMIiA9PSAibmdyb2siICYmIC16ICIke05HUk9LX0RPTUFJTjotfSIgXV07IHRoZW4KICB3YXJuICJuZ3JvayBmcmVlIHRpZXIgc2hvd3MgYSB3YXJuaW5nIHBhZ2Ugb24gdGhlIGZpcnN0IGJyb3dzZXIgaGl0LiIKICB3YXJuICJDbGljayB0aHJvdWdoIGl0LCBvciB0aGUgZWRpdG9yJ3MgSlMgYXNzZXRzIGNvbWUgYmFjayBhcyBIVE1MIChibGFuayBwYWdlKS4iCiAgd2FybiAiU2V0IE5HUk9LX0RPTUFJTj08cmVzZXJ2ZWQtZG9tYWluPiB0byBrZWVwIG9uZSBob3N0bmFtZSBhY3Jvc3MgcmVzdGFydHMsIgogIHdhcm4gIm9yIENTX1RVTk5FTD1jbG91ZGZsYXJlZCBmb3Igbm8gaW50ZXJzdGl0aWFsIGF0IGFsbC4iCmZpCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBmb3JlZ3JvdW5kIC0tLS0KaWYgKChGT1JFR1JPVU5EKSk7IHRoZW4KICBsb2cgImZvcmVncm91bmQgbW9kZTsgQ3RybC1DIHRvIHN0b3AiCiAgd2hpbGUgOjsgZG8KICAgIGlmICEga2lsbCAtMCAiJENTX1BJRCIgMj4vZGV2L251bGw7IHRoZW4KICAgICAgd2FybiAiY29kZS1zZXJ2ZXIgZXhpdGVkIgogICAgICBleGl0IDEKICAgIGZpCiAgICBpZiBbWyAtbiAiJFRVTl9QSUQiIF1dICYmICEga2lsbCAtMCAiJFRVTl9QSUQiIDI+L2Rldi9udWxsOyB0aGVuCiAgICAgIHdhcm4gInR1bm5lbCBleGl0ZWQiCiAgICAgIGV4aXQgMQogICAgZmkKICAgIHNsZWVwIDIKICBkb25lCmZpCg=='
    ),
    'supervise.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc3VwZXJ2aXNlLnNoIOKAlCBrZWVwIHRoZSBzdGFjayBhbGl2ZSBhbmQgc3RvcCBpdCBiZWZvcmUgQ29sYWIgZG9lcy4KIwojICAgYmFzaCBjb2xhYi9zdXBlcnZpc2Uuc2gKIyAgIENTX0lETEVfVElNRU9VVD05MDAgQ1NfTUFYX1JVTlRJTUU9MjE2MDAgYmFzaCBjb2xhYi9zdXBlcnZpc2Uuc2gKIwojIFR3byBqb2JzOgojICAgMS4gcmVzdGFydCBzZXJ2ZS5zaCB3aXRoIGV4cG9uZW50aWFsIGJhY2tvZmYgaWYgY29kZS1zZXJ2ZXIgb3IgdGhlIHR1bm5lbCBkaWVzCiMgICAyLiBzaHV0IHRoZSBzdGFjayBkb3duIHdoZW4gbm9ib2R5IGlzIGNvbm5lY3RlZCwgb3Igd2hlbiB0aGUgcnVudGltZSBoYXMgYmVlbgojICAgICAgaGVsZCBmb3IgdG9vIGxvbmcKIwojICgyKSBpcyB0aGUgaG9uZXN0IGhhbGYgb2YgImxlc3Mgbm90aWNlYWJsZSI6IGEgR1BVIFZNIGhlbGQgb3BlbiBmb3IgaG91cnMgd2l0aAojIG5vIGludGVyYWN0aXZlIHRyYWZmaWMgaXMgdGhlIHBhdHRlcm4gdGhhdCBnZXRzIGFjY291bnRzIHRocm90dGxlZC4gUmVsZWFzaW5nCiMgaXQgd2hlbiBpZGxlIGlzIGJvdGggYmV0dGVyIG1hbm5lcnMgYW5kIGZld2VyIHNlc3Npb25zIHRvIGJlIGZsYWdnZWQuCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKUE9SVD0iJHtDU19QT1JUOi04MDgwfSIKSURMRV9USU1FT1VUPSIke0NTX0lETEVfVElNRU9VVDotMTgwMH0iICAgICAjIDMwIG1pbiB3aXRoIHplcm8gY29ubmVjdGlvbnMKTUFYX1JVTlRJTUU9IiR7Q1NfTUFYX1JVTlRJTUU6LTM5NjAwfSIgICAgICAjIDExIGgsIHVuZGVyIENvbGFiJ3MgaGFyZCBjYXAKSEVBTFRIWV9BRlRFUj0iJHtDU19IRUFMVEhZX0FGVEVSOi02MDB9IiAgICAjIHVwdGltZSB0aGF0IHJlc2V0cyB0aGUgYmFja29mZgpNQVhfQkFDS09GRj0iJHtDU19NQVhfQkFDS09GRjotNjB9IgpNQVhfUkVTVEFSVFM9IiR7Q1NfTUFYX1JFU1RBUlRTOi0yMH0iCgpbWyAiJElETEVfVElNRU9VVCIgPX4gXlswLTldKyQgXV0gfHwgZGllICJDU19JRExFX1RJTUVPVVQgbXVzdCBiZSBhIG51bWJlciBvZiBzZWNvbmRzIgpbWyAiJE1BWF9SVU5USU1FIiA9fiBeWzAtOV0rJCBdXSAgfHwgZGllICJDU19NQVhfUlVOVElNRSBtdXN0IGJlIGEgbnVtYmVyIG9mIHNlY29uZHMiCgpjc19uZWVkIGNvZGUtc2VydmVyICJydW46IGJhc2ggY29sYWIvaW5zdGFsbC5zaCIKY3NfbmVlZCBjdXJsCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpjc19sb2NrIHN1cGVydmlzZQp0cmFwICdjc191bmxvY2snIEVYSVQKCnN0YWNrX2FsaXZlKCkgewogIGxvY2FsIGNzX3BpZAogIGNzX3BpZD0iJChjc19yZWFkX3BpZCAiJHtTRH0vY3MucGlkIikiCiAgW1sgLW4gIiRjc19waWQiIF1dICYmIGtpbGwgLTAgIiRjc19waWQiIDI+L2Rldi9udWxsCn0KCiMgUmV0dXJuczogMCA9IHN0aWxsIHJ1bm5pbmcsIDEwID0gaWRsZSBzaHV0ZG93biwgMTEgPSBtYXggcnVudGltZSByZWFjaGVkLAojICAgICAgICAgIDEgPSBzb21ldGhpbmcgZGllZCAoY2FsbGVyIHJlc3RhcnRzKQp3YXRjaF9zdGFjaygpIHsKICBsb2NhbCBzdGFydGVkPSRTRUNPTkRTIGxhc3RfYWN0aXZlPSRTRUNPTkRTIGNvbm5zCiAgd2hpbGUgc3RhY2tfYWxpdmU7IGRvCiAgICBzbGVlcCA1CiAgICBjb25ucz0iJChjc19hY3RpdmVfY29ubnMgIiRQT1JUIikiCiAgICBpZiBbWyAiJHtjb25uczotMH0iIC1ndCAwIF1dOyB0aGVuCiAgICAgIGxhc3RfYWN0aXZlPSRTRUNPTkRTCiAgICBmaQogICAgaWYgKChTRUNPTkRTIC0gbGFzdF9hY3RpdmUgPiBJRExFX1RJTUVPVVQpKTsgdGhlbgogICAgICBsb2cgIm5vIGNvbm5lY3Rpb25zIGZvciAke0lETEVfVElNRU9VVH1zIOKAlCBzaHV0dGluZyBkb3duIgogICAgICBiYXNoICIke0hFUkV9L3N0b3Auc2giIHx8IHRydWUKICAgICAgcmV0dXJuIDEwCiAgICBmaQogICAgaWYgKChTRUNPTkRTIC0gc3RhcnRlZCA+IE1BWF9SVU5USU1FKSk7IHRoZW4KICAgICAgbG9nICJtYXggcnVudGltZSAke01BWF9SVU5USU1FfXMgcmVhY2hlZCDigJQgc2h1dHRpbmcgZG93biIKICAgICAgYmFzaCAiJHtIRVJFfS9zdG9wLnNoIiB8fCB0cnVlCiAgICAgIHJldHVybiAxMQogICAgZmkKICBkb25lCiAgcmV0dXJuIDEKfQoKcmVzdGFydHM9MApkZWxheT0yCgp3aGlsZSA6OyBkbwogIHQwPSRTRUNPTkRTCgogIGlmIGJhc2ggIiR7SEVSRX0vc2VydmUuc2giICIkQCIgOT4mLTsgdGhlbgogICAgcmM9MAogICAgd2F0Y2hfc3RhY2sgfHwgcmM9JD8KICAgIGNhc2UgIiRyYyIgaW4KICAgICAgMTB8MTEpIGxvZyAic3RvcHBlZCBjbGVhbmx5IjsgZXhpdCAwIDs7CiAgICBlc2FjCiAgICBbWyAiJHJjIiAtZXEgMCBdXSAmJiBjb250aW51ZQogIGVsc2UKICAgIHdhcm4gInNlcnZlLnNoIGZhaWxlZCB0byBzdGFydCIKICBmaQoKICB1cHRpbWU9JCgoU0VDT05EUyAtIHQwKSkKICBpZiAoKHVwdGltZSA+PSBIRUFMVEhZX0FGVEVSKSk7IHRoZW4KICAgIGRlbGF5PTIKICAgIHJlc3RhcnRzPTAKICBlbHNlCiAgICByZXN0YXJ0cz0kKChyZXN0YXJ0cyArIDEpKQogICAgZGVsYXk9JCgoZGVsYXkgKiAyKSkKICAgICgoZGVsYXkgPiBNQVhfQkFDS09GRikpICYmIGRlbGF5PSRNQVhfQkFDS09GRgogIGZpCgogIGlmICgocmVzdGFydHMgPiBNQVhfUkVTVEFSVFMpKTsgdGhlbgogICAgY3NfZHVtcF9sb2cgIiR7U0R9L2NvZGUtc2VydmVyLmxvZyIgNDAKICAgIGRpZSAiJHtyZXN0YXJ0c30gZmFpbGVkIHJlc3RhcnRzIGluIGEgcm93OyBnaXZpbmcgdXAiCiAgZmkKCiAgd2FybiAicmVzdGFydGluZyBpbiAke2RlbGF5fXMgKHVwdGltZSAke3VwdGltZX1zLCBhdHRlbXB0ICR7cmVzdGFydHN9KSIKICBzbGVlcCAiJGRlbGF5Igpkb25lCg=='
    ),
    'stop.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc3RvcC5zaCDigJQgc3RvcCBjb2RlLXNlcnZlciBhbmQgYW55IHR1bm5lbCBzdGFydGVkIGJ5IHNlcnZlLnNoLgojCiMgICBiYXNoIGNvbGFiL3N0b3Auc2gKIwpzZXQgLUVldW8gcGlwZWZhaWwKSEVSRT0iJChjZCAtLSAiJChkaXJuYW1lIC0tICIke0JBU0hfU09VUkNFWzBdfSIpIiAmJiBwd2QgLVApIgojIHNoZWxsY2hlY2sgc291cmNlPWxpYi9jb21tb24uc2gKc291cmNlICIke0hFUkV9L2xpYi9jb21tb24uc2giCgpTRD0iJChjc19zdGF0ZV9kaXIpIgoKY3Nfc3RvcF9waWRmaWxlICIke1NEfS9jcy5waWQiIGNvZGUtc2VydmVyCmNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vbmdyb2sucGlkIiBuZ3Jvawpjc19zdG9wX3BpZGZpbGUgIiR7U0R9L2Nsb3VkZmxhcmVkLnBpZCIgY2xvdWRmbGFyZWQKCiMgRmFsbGJhY2sgZm9yIGEgcnVuIHdob3NlIHBpZGZpbGUgd2FzIGxvc3QuIFRoZSBwYXR0ZXJucyBhcmUgYW5jaG9yZWQgdG8gdGhlCiMgZnVsbCBhcmd2IG9uIHB1cnBvc2U6IHRoaXMgcmVwb3NpdG9yeSdzIGRpcmVjdG9yeSBpcyBpdHNlbGYgbmFtZWQKIyAiY29kZS1zZXJ2ZXIiLCBzbyBhIGJhcmUgYHBraWxsIC1mIGNvZGUtc2VydmVyYCB3b3VsZCBhbHNvIG1hdGNoIOKAlCBhbmQga2lsbCDigJQKIyB0aGUgc2hlbGwgdGhhdCBpbnZva2VkIHRoaXMgc2NyaXB0Lgpwa2lsbCAtZiAnY29kZS1zZXJ2ZXIgLS1iaW5kLWFkZHInIDI+L2Rldi9udWxsIHx8IHRydWUKcGtpbGwgLWYgJ25ncm9rIGh0dHAnIDI+L2Rldi9udWxsIHx8IHRydWUKcGtpbGwgLWYgJ2Nsb3VkZmxhcmVkIHR1bm5lbCcgMj4vZGV2L251bGwgfHwgdHJ1ZQoKcm0gLWYgLS0gIiR7U0R9L2NzLnBpZCIgIiR7U0R9L25ncm9rLnBpZCIgIiR7U0R9L2Nsb3VkZmxhcmVkLnBpZCIKb2sgInN0b3BwZWQiCg=='
    ),
    'doctor.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvZG9jdG9yLnNoIOKAlCByZXBvcnQgdGhlIHN0YXRlIG9mIHRoZSBDb2xhYiBlbnZpcm9ubWVudCBhbmQgdGhlIHN0YWNrLgojCiMgICBiYXNoIGNvbGFiL2RvY3Rvci5zaAojCiMgUmVhZC1vbmx5OiBjaGFuZ2VzIG5vdGhpbmcuIFJ1biBpdCBiZWZvcmUgZmlsaW5nIGEgIml0IGRvZXNuJ3Qgd29yayIgbm90ZS4KIwpzZXQgLUVldW8gcGlwZWZhaWwKSEVSRT0iJChjZCAtLSAiJChkaXJuYW1lIC0tICIke0JBU0hfU09VUkNFWzBdfSIpIiAmJiBwd2QgLVApIgojIHNoZWxsY2hlY2sgc291cmNlPWxpYi9jb21tb24uc2gKc291cmNlICIke0hFUkV9L2xpYi9jb21tb24uc2giCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpQT1JUPSIke0NTX1BPUlQ6LTgwODB9Igpwcm9ibGVtcz0wCgpiYWQoKSB7IHdhcm4gIiQqIjsgcHJvYmxlbXM9JCgocHJvYmxlbXMgKyAxKSk7IH0KCmt2KCkgeyBwcmludGYgJyAgICUtMjJzICVzXG4nICIkMSIgIiQyIjsgfQoKc2VjdGlvbigpIHsgcHJpbnRmICdcbiVzXG4nICIkKHByaW50ZiAnXDAzM1sxbSVzXDAzM1swbScgIiQqIikiOyB9CgpzZWN0aW9uICJob3N0IgprdiAiYXJjaGl0ZWN0dXJlIiAiJCh1bmFtZSAtbSkiCmt2ICJvcyIgIiQoY3Nfb3NfZmllbGQgUFJFVFRZX05BTUUpIgprdiAia2VybmVsIiAiJCh1bmFtZSAtcikiCmt2ICJ1c2VyIiAiJChpZCAtdW4pICh1aWQgJChpZCAtdSkpIgprdiAicHl0aG9uIiAiJChjb21tYW5kIC12IHB5dGhvbjMgPi9kZXYvbnVsbCAyPiYxICYmIHB5dGhvbjMgLVYgMj4mMSB8fCBlY2hvIG1pc3NpbmcpIgprdiAiYmFzaCIgIiRCQVNIX1ZFUlNJT04iCgpzZWN0aW9uICJyZXNvdXJjZXMiCm1lbV9nYj0iJChhd2sgJy9NZW1Ub3RhbC8geyBwcmludGYgIiUuMWYiLCAkMiAvIDEwNDg1NzYgfScgL3Byb2MvbWVtaW5mbyAyPi9kZXYvbnVsbCB8fCBlY2hvICc/JykiCmt2ICJtZW1vcnkiICIke21lbV9nYn0gR0IiCmlmIFtbIC1kIC9jb250ZW50IF1dOyB0aGVuCiAga3YgImRpc2sgL2NvbnRlbnQiICIkKGRmIC1oIC9jb250ZW50IDI+L2Rldi9udWxsIHwgYXdrICdOUj09MiB7IHByaW50ICQ0ICIgZnJlZSBvZiAiICQyIH0nKSIKZmkKa3YgInN0YXRlIGRpciIgIiR7U0R9IgprdiAic3RhdGUgc2l6ZSIgIiQoZHUgLXNoICIkU0QiIDI+L2Rldi9udWxsIHwgYXdrICd7IHByaW50ICQxIH0nIHx8IGVjaG8gJzAnKSIKCnNlY3Rpb24gImNvZGUtc2VydmVyIgppZiBjb21tYW5kIC12IGNvZGUtc2VydmVyID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJwYXRoIiAiJChjb21tYW5kIC12IGNvZGUtc2VydmVyKSIKICBrdiAidmVyc2lvbiIgIiQoY29kZS1zZXJ2ZXIgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSB8fCBlY2hvICc/JykiCiAgW1sgLWYgIiR7U0R9Ly5pbnN0YWxsZWQtdmVyc2lvbiIgXV0gJiYga3YgImluc3RhbGxlZCB0YWciICIkKGNhdCAiJHtTRH0vLmluc3RhbGxlZC12ZXJzaW9uIikiCmVsc2UKICBiYWQgImNvZGUtc2VydmVyIGlzIG5vdCBvbiBQQVRIIChydW46IGJhc2ggY29sYWIvaW5zdGFsbC5zaCkiCmZpCgpzZWN0aW9uICJ0dW5uZWxzIgppZiBjb21tYW5kIC12IG5ncm9rID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJuZ3JvayIgIiQobmdyb2sgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCiAgaWYgbmdyb2sgY29uZmlnIGNoZWNrID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAga3YgIm5ncm9rIGF1dGh0b2tlbiIgImNvbmZpZ3VyZWQiCiAgZWxzZQogICAgYmFkICJuZ3JvayBhdXRodG9rZW4gbm90IGNvbmZpZ3VyZWQgKG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuIDx0b2tlbj4pIgogIGZpCmVsc2UKICBrdiAibmdyb2siICJub3QgaW5zdGFsbGVkIgpmaQppZiBjb21tYW5kIC12IGNsb3VkZmxhcmVkID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJjbG91ZGZsYXJlZCIgIiQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKZWxzZQogIGt2ICJjbG91ZGZsYXJlZCIgIm5vdCBpbnN0YWxsZWQiCmZpCgpzZWN0aW9uICJydW50aW1lIHN0YXRlIgppZiBjc19hbGl2ZSAiJHtTRH0vY3MucGlkIjsgdGhlbgogIGt2ICJjb2RlLXNlcnZlciIgInJ1bm5pbmcgKHBpZCAkKGNzX3JlYWRfcGlkICIke1NEfS9jcy5waWQiKSkiCmVsc2UKICBrdiAiY29kZS1zZXJ2ZXIiICJub3QgcnVubmluZyIKZmkKaWYgY3NfYWxpdmUgIiR7U0R9L25ncm9rLnBpZCI7IHRoZW4KICBrdiAibmdyb2siICJydW5uaW5nIChwaWQgJChjc19yZWFkX3BpZCAiJHtTRH0vbmdyb2sucGlkIikpIgpmaQppZiBjc19hbGl2ZSAiJHtTRH0vY2xvdWRmbGFyZWQucGlkIjsgdGhlbgogIGt2ICJjbG91ZGZsYXJlZCIgInJ1bm5pbmcgKHBpZCAkKGNzX3JlYWRfcGlkICIke1NEfS9jbG91ZGZsYXJlZC5waWQiKSkiCmZpCgprdiAicG9ydCAke1BPUlR9IiAiJChjc19wb3J0X2luX3VzZSAiJFBPUlQiICYmIGVjaG8gJ2luIHVzZScgfHwgZWNobyAnZnJlZScpIgprdiAiZXN0YWJsaXNoZWQgY29ubnMiICIkKGNzX2FjdGl2ZV9jb25ucyAiJFBPUlQiKSIKCmlmIGN1cmwgLWZzUyAtLW1heC10aW1lIDIgLS1ub3Byb3h5ICcqJyAiaHR0cDovLzEyNy4wLjAuMToke1BPUlR9L2hlYWx0aHoiID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJoZWFsdGh6IiAib2siCmVsc2UKICBrdiAiaGVhbHRoeiIgIm5vdCByZXNwb25kaW5nIgpmaQoKaWYgW1sgLXMgIiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIgXV07IHRoZW4KICBzZWN0aW9uICJjcmVkZW50aWFscyAoZnJvbSB0aGUgbGFzdCBzZXJ2ZS5zaCBydW4pIgogICMgUGFzc3dvcmQgaW50ZW50aW9uYWxseSBub3QgcHJpbnRlZCBoZXJlOyByZWFkIHRoZSAwNjAwIGZpbGUgaWYgeW91IG5lZWQgaXQuCiAgc2VkIC1lICdzL15wYXNzd29yZD0uKi9wYXNzd29yZD08aGlkZGVuOyBzZWUgdGhlIGZpbGU+LycgIiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIgXAogICAgfCBzZWQgJ3MvXi8gICAvJwpmaQoKc2VjdGlvbiAibG9ncyIKZm9yIGYgaW4gY29kZS1zZXJ2ZXIubG9nIG5ncm9rLmxvZyBjbG91ZGZsYXJlZC5sb2cgZXh0ZW5zaW9ucy5sb2c7IGRvCiAgaWYgW1sgLWYgIiR7U0R9LyR7Zn0iIF1dOyB0aGVuCiAgICBwcmludGYgJyAgICUtMjJzICVzIGJ5dGVzXG4nICIkZiIgIiQod2MgLWMgPCIke1NEfS8ke2Z9IiB8IHRyIC1kICcgJykiCiAgZmkKZG9uZQoKcHJpbnRmICdcbicKaWYgKChwcm9ibGVtcyA9PSAwKSk7IHRoZW4KICBvayAibm8gcHJvYmxlbXMgZGV0ZWN0ZWQiCmVsc2UKICB3YXJuICIke3Byb2JsZW1zfSBwcm9ibGVtKHMpIGRldGVjdGVkIgpmaQo='
    ),
    'README.md': (
        'IyBjb2RlLXNlcnZlciBvbiBHb29nbGUgQ29sYWIKClJ1bnMgY29kZS1zZXJ2ZXIgaW4gYSBDb2xhYiBWTSBhbmQgZXhwb3NlcyBpdCBvdmVyIGEgdHVubmVsLgoKKipObyBwYXRjaGVzIHRvIGNvZGUtc2VydmVyLioqIEl0IGlzIGEgcGxhaW4gTm9kZSBIVFRQIHNlcnZlcjsgbm90aGluZyBpbiBgc3JjL2AKb3IgYHBhdGNoZXMvYCBuZWVkcyB0byBjaGFuZ2UuIFRoZXNlIHNjcmlwdHMgb25seSBwaWNrIGZsYWdzIGFwcHJvcHJpYXRlIHRvIGFuCmVwaGVtZXJhbCBDb2xhYiBWTSwgaW5zdGFsbCB0aGUgcnVudGltZSBsaWJyYXJpZXMsIGFuZCBtYW5hZ2UgdGhlIHR1bm5lbC4KCmBgYApjb2xhYi8K4pSc4pSA4pSAIGluc3RhbGwuc2ggICAgICAgIyBsaWJyYXJpZXMgKyBjb2RlLXNlcnZlciB0YXJiYWxsIChjYWNoZWQpICsgdHVubmVsIGFnZW50CuKUnOKUgOKUgCBzZXJ2ZS5zaCAgICAgICAgICMgc3RhcnQgY29kZS1zZXJ2ZXIsIHdhaXQgZm9yIGhlYWx0aCwgb3BlbiB0dW5uZWwsIHJlcG9ydArilJzilIDilIAgc3VwZXJ2aXNlLnNoICAgICAjIHJlc3RhcnQgd2l0aCBiYWNrb2ZmICsgaWRsZSAvIG1heC1ydW50aW1lIHdhdGNoZG9nCuKUnOKUgOKUgCBzdG9wLnNoICAgICAgICAgICMgc3RvcCBldmVyeXRoaW5nCuKUnOKUgOKUgCBkb2N0b3Iuc2ggICAgICAgICMgcmVhZC1vbmx5IGRpYWdub3N0aWNzCuKUnOKUgOKUgCBsaWIvY29tbW9uLnNoICAgICMgc2hhcmVkIGhlbHBlcnMgKGxvZ2dpbmcsIGxvY2tzLCBwaWRmaWxlcywgcHJvYmVzKQrilJTilIDilIAgY29kZS1zZXJ2ZXIuaXB5bmIKYGBgCgojIyBXaGF0IGEgInRyYWNlIiBhY3R1YWxseSBpcywgYW5kIHdoYXQgYSB3cmFwcGVyIGNhbiBkbyBhYm91dCBpdAoKV29ydGggcmVhZGluZyBiZWZvcmUgYXNraW5nIGZvciBwcm9jZXNzIHJlbmFtaW5nLCBiZWNhdXNlIHRoZSBsYXllcnMgYXJlIG5vdAplcXVhbGx5IHJlYWNoYWJsZToKCnwgTGF5ZXIgfCBPYnNlcnZlZCBieSB8IENoYW5nZWFibGUgYnkgYSB3cmFwcGVyPyB8CnwtLS18LS0tfC0tLXwKfCBgL3Byb2MvPHBpZD4vY21kbGluZWAsIHByb2Nlc3MgbmFtZSB8IGFueXRoaW5nIGluc2lkZSB0aGUgVk0gfCB5ZXMsIHRyaXZpYWxseSDigJQgYW5kIGl0IGRvZXMgbm90IG1hdHRlcjogQ29sYWIncyBlbmZvcmNlbWVudCBpcyBub3QgYSBwZXItcHJvY2VzcyBzY2FubmVyLiBjb2RlLXNlcnZlcidzIG93biBjaGlsZHJlbiAoYG5vZGVgLCBleHRlbnNpb24gaG9zdHMsIGV2ZXJ5IHRlcm1pbmFsIHlvdSBvcGVuKSBrZWVwIHRoZSByZWFsIG5hbWVzLCBzbyBhIHJlbmFtZWQgcGFyZW50IGlzICppbmNvbnNpc3RlbnQqLCB3aGljaCBpcyBhIHdvcnNlIHNpZ25hbCB0aGFuIGFuIGhvbmVzdCBvbmUuIHwKfCBmaWxlc3lzdGVtIHdyaXRlcyB8IHRoZSBWTSBpdHNlbGYgfCB5ZXMsIHBvaW50bGVzcyDigJQgdGhlIGRpc2sgaXMgd2lwZWQgcGVyIHJ1bnRpbWUgfAp8IGVncmVzcyBTTkkgLyBob3N0bmFtZSAoYCoubmdyb2stZnJlZS5hcHBgKSB8IHRoZSBuZXR3b3JrIHBhdGggfCBvbmx5IGJ5IHVzaW5nIHlvdXIgb3duIGRvbWFpbiBvbiBhIG5hbWVkIHR1bm5lbC4gVGhhdCBpcyBub3QgZGlzZ3Vpc2UsIGl0IGlzIGhvc3Rpbmcg4oCUIHNlZSBiZWxvdy4gfAp8IHR1bm5lbC1zaWRlIEhUVFAvV1Mgc2hhcGUgKGAvd2Vic29ja2V0YCwgYXNzZXQgcGF0aHMpIHwgbmdyb2sgLyBDbG91ZGZsYXJlIGVkZ2UsIG5vdCBHb29nbGUgfCBubyByZWFzb24gdG8gY2FyZSB8CnwgcnVudGltZSBiZWhhdmlvdXI6IGlkbGUgR1BVIFZNLCBzZXNzaW9uIGxlbmd0aCwgemVybyBleGVjdXRlZCBjZWxscywgbWVtb3J5L0NQVSBwcm9maWxlIHwgR29vZ2xlLCBhY2NvdW50IGxldmVsIHwgKipubyoqIOKAlCBhbmQgdGhpcyBpcyB0aGUgbGF5ZXIgdGhhdCBnZXRzIGFjdGlvbmVkIHwKfCBhY2NvdW50IC8gc2Vzc2lvbiBjb3VudCAvIElQIHJlcHV0YXRpb24gfCBHb29nbGUgfCBubyB8CgpTbyB0aGUgdHdvIGxheWVycyBhIHdyYXBwZXIgKmNhbiogdG91Y2ggYXJlIHRoZSB0d28gbm9ib2R5IGVuZm9yY2VzIG9uLCBhbmQgdGhlCm9uZSB0aGF0IG1hdHRlcnMgaXMgdW50b3VjaGVkIGJ5IGFueSBzY3JpcHQuIFdoYXQgdGhpcyB0b29saW5nIGRvZXMgaW5zdGVhZCBpcwphZGRyZXNzIHRoZSB1bmRlcmx5aW5nIGNvbXBsYWludHMgcHJvcGVybHk6CgoqICoqcm90YXRpbmcgVVJMKiog4oaSIG5hbWVkIENsb3VkZmxhcmUgdHVubmVsIG9uIHlvdXIgb3duIGRvbWFpbiwgc3RhYmxlIGFjcm9zcwogIHJlc3RhcnRzIChgQ0ZfVE9LRU5gICsgYENGX1RVTk5FTF9VUkxgKSwgbm8gaW50ZXJzdGl0aWFsIHBhZ2UKKiAqKmJsYW5rIHBhZ2Ugb24gbmdyb2sqKiDihpIgcmVzZXJ2ZWQgbmdyb2sgZG9tYWluIChgTkdST0tfRE9NQUlOYCkKKiAqKlZNIGhlbGQgb3BlbiBmb3IgaG91cnMgZm9yIG5vdGhpbmcqKiDihpIgYHN1cGVydmlzZS5zaGAgaWRsZSBhbmQgbWF4LXJ1bnRpbWUKICB3YXRjaGRvZ3MuIFRoaXMgaXMgYWxzbyB0aGUgaG9uZXN0IGZpeCBmb3IgImxlc3Mgbm90aWNlYWJsZSI6IGEgR1BVIFZNIGlkbGluZwogIHdpdGggbm8gaW50ZXJhY3RpdmUgdHJhZmZpYyBpcyB0aGUgYWN0dWFsIHBhdHRlcm4gdGhhdCBnZXRzIHRocm90dGxlZCwgYW5kCiAgcmVsZWFzaW5nIGl0IGNvc3RzIG5vdGhpbmcuCgpUcmFmZmljIG9yIHByb2Nlc3MgcmVzcG9vZmluZyBpcyBkZWxpYmVyYXRlbHkgbm90IGltcGxlbWVudGVkLgoKIyMgUXVpY2sgc3RhcnQKCmBgYGJhc2gKZXhwb3J0IE5HUk9LX1RPS0VOPS4uLiAgICAgICAgIyBkYXNoYm9hcmQubmdyb2suY29tL2dldC1zdGFydGVkL3lvdXItYXV0aHRva2VuCmJhc2ggY29sYWIvaW5zdGFsbC5zaApiYXNoIGNvbGFiL3NlcnZlLnNoCmBgYAoKIyMgVHVubmVscwoKIyMjIG5ncm9rIChkZWZhdWx0KQoKYGBgYmFzaApiYXNoIGNvbGFiL3NlcnZlLnNoICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGF1dG8tc2VsZWN0cyBhIHR1bm5lbApDU19UVU5ORUw9bmdyb2sgYmFzaCBjb2xhYi9zZXJ2ZS5zaApOR1JPS19ET01BSU49bXktZGV2Lm5ncm9rLWZyZWUuYXBwIGJhc2ggY29sYWIvc2VydmUuc2ggICAjIHJlc2VydmVkLCBzdGFibGUKTkdST0tfQkFTSUNfQVVUSD0ndXNlcjpwYXNzJyBiYXNoIGNvbGFiL3NlcnZlLnNoICAgICAgICAgIyBnYXRlIGF0IHRoZSBlZGdlCmBgYAoKRnJlZSB0aWVyIHNob3dzIGEgIllvdSBhcmUgYWJvdXQgdG8gdmlzaXQiIHBhZ2Ugb24gdGhlIGZpcnN0IGJyb3dzZXIgaGl0LiBDbGljawp0aHJvdWdoIGl0IGJlZm9yZSB0aGUgZWRpdG9yIGxvYWRzLCBvdGhlcndpc2UgdGhlIEpTIGFzc2V0cyBhcmUgc2VydmVkIGFzIEhUTUwKYW5kIHlvdSBnZXQgYSBibGFuayBwYWdlLiBGcmVlIHRpZXIgYWxzbyBhbGxvd3Mgb25lIGFnZW50IHNlc3Npb24gYW5kIHJvdGF0ZXMKdGhlIGhvc3RuYW1lIHVubGVzcyB5b3UgcmVzZXJ2ZSBhIGRvbWFpbi4KCiMjIyBjbG91ZGZsYXJlZCAobm8gaW50ZXJzdGl0aWFsLCBzdXBwb3J0cyBhIHJlYWwgZG9tYWluKQoKYGBgYmFzaApXSVRIX0NMT1VERkxBUkVEPTEgYmFzaCBjb2xhYi9pbnN0YWxsLnNoCmJhc2ggY29sYWIvc2VydmUuc2ggLS10dW5uZWw9Y2xvdWRmbGFyZWQgICAgICAgICAgICAjIGh0dHBzOi8vPHJhbmQ+LnRyeWNsb3VkZmxhcmUuY29tCmBgYAoKTmFtZWQgdHVubmVsLCBzdGFibGUgaG9zdG5hbWUsIG5vIHdhcm5pbmcgcGFnZToKCmBgYGJhc2gKY2xvdWRmbGFyZWQgdHVubmVsIGxvZ2luCmNsb3VkZmxhcmVkIHR1bm5lbCBjcmVhdGUgY29sYWIKY2xvdWRmbGFyZWQgdHVubmVsIHJvdXRlIGRucyBjb2xhYiBjb2RlLmV4YW1wbGUuY29tCiMgdGhlbiwgcGVyIHNlc3Npb246CkNGX1RPS0VOPTx0dW5uZWwtdG9rZW4+IENGX1RVTk5FTF9VUkw9aHR0cHM6Ly9jb2RlLmV4YW1wbGUuY29tIFwKICBiYXNoIGNvbGFiL3NlcnZlLnNoIC0tdHVubmVsPWNsb3VkZmxhcmVkCmBgYAoKVGhpcyBpcyB0aGUgZHVyYWJsZSBvcHRpb24uIE5vdGhpbmcgaXMgYmVpbmcgaGlkZGVuOiB0aGUgdHJhZmZpYyBpcyBhIG5vcm1hbApUTFMgY29ubmVjdGlvbiB0byBhIGRvbWFpbiB5b3UgY29udHJvbC4KCiMjIyBOb3QgYW4gb3B0aW9uOiBDb2xhYidzIG93biBwb3J0IHByb3h5CgpgZ29vZ2xlLmNvbGFiLm91dHB1dC5zZXJ2ZV9rZXJuZWxfcG9ydF9hc193aW5kb3coKWAgY2FuIGV4cG9zZSBhIHBvcnQsIGJ1dCB0aGUKcHJveHkgaXMgYnVpbHQgYXJvdW5kIEp1cHl0ZXIga2VybmVsIGNoYW5uZWxzIGFuZCBkb2VzIG5vdCByZWxpYWJseSBjYXJyeSB0aGUKbG9uZy1saXZlZCBiaWRpcmVjdGlvbmFsIFdlYlNvY2tldCB0aGUgVlMgQ29kZSB3ZWIgY2xpZW50IG5lZWRzLiBUaGUgZWRpdG9yIG1heQpsb2FkIGFuZCB0aGVuIGxvc2UgaXRzIHRlcm1pbmFsLiBUaGF0IGlzIHdoeSBldmVyeSB3b3JraW5nIHNldHVwIHVzZXMgYSB0dW5uZWwuCgojIyBTdXBlcnZpc2lvbgoKYGBgYmFzaApiYXNoIGNvbGFiL3N1cGVydmlzZS5zaApgYGAKCiogcmVzdGFydHMgYHNlcnZlLnNoYCB3aXRoIGV4cG9uZW50aWFsIGJhY2tvZmYgKDIgcyDihpIgNjAgcykgYWZ0ZXIgYSBjcmFzaAoqIHJlc2V0cyB0aGUgYmFja29mZiBvbmNlIHRoZSBzdGFjayBoYXMgYmVlbiB1cCBmb3IgYENTX0hFQUxUSFlfQUZURVJgIHNlY29uZHMKKiBzaHV0cyBkb3duIGFmdGVyIGBDU19JRExFX1RJTUVPVVRgIHNlY29uZHMgd2l0aCB6ZXJvIGVzdGFibGlzaGVkIGNvbm5lY3Rpb25zCiogc2h1dHMgZG93biBhZnRlciBgQ1NfTUFYX1JVTlRJTUVgIHNlY29uZHMsIGFoZWFkIG9mIENvbGFiJ3Mgb3duIGhhcmQgY2FwCgojIyBDb25maWd1cmF0aW9uCgp8IFZhcmlhYmxlIHwgRGVmYXVsdCB8IE1lYW5pbmcgfAp8LS0tfC0tLXwtLS18CnwgYE5HUk9LX1RPS0VOYCB8IOKAlCB8IG5ncm9rIGF1dGh0b2tlbiwgb3IgYWRkIGl0IG9uY2Ugd2l0aCBgbmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW5gIHwKfCBgTkdST0tfRE9NQUlOYCB8IOKAlCB8IHJlc2VydmVkIGRvbWFpbiDihpIgc3RhYmxlIGhvc3RuYW1lLCBubyByb3RhdGlvbiB8CnwgYE5HUk9LX0JBU0lDX0FVVEhgIHwg4oCUIHwgYHVzZXI6cGFzc2AsIHJlamVjdHMgc2Nhbm5lcnMgYXQgbmdyb2sncyBlZGdlIHwKfCBgQ0ZfVE9LRU5gIHwg4oCUIHwgQ2xvdWRmbGFyZSBuYW1lZC10dW5uZWwgdG9rZW4gfAp8IGBDRl9UVU5ORUxfVVJMYCB8IOKAlCB8IHB1YmxpYyBob3N0bmFtZSB0byByZXBvcnQgZm9yIGEgbmFtZWQgdHVubmVsIHwKfCBgQ1NfVFVOTkVMYCB8IGBhdXRvYCB8IGBhdXRvYCBcfCBgbmdyb2tgIFx8IGBjbG91ZGZsYXJlZGAgXHwgYG5vbmVgLiBgYXV0b2AgPSBuZ3JvayB3aGVuIGl0IGhhcyBjcmVkZW50aWFscywgZWxzZSBhIGNsb3VkZmxhcmVkIHF1aWNrIHR1bm5lbCAobm8gYWNjb3VudCBuZWVkZWQpIHwKfCBgQ1NfUE9SVGAgfCBgODA4MGAgfCBsb2NhbCBwb3J0IChuZXZlciBwdWJsaWM7IHRoZSBhZ2VudCBydW5zIGluIHRoZSBzYW1lIFZNKSB8CnwgYENTX1BBU1NXT1JEYCB8IGdlbmVyYXRlZCB8IGZpeGVkIHBhc3N3b3JkOyBvdGhlcndpc2UgcGVyc2lzdGVkIHRvIHRoZSBzdGF0ZSBkaXIgfAp8IGBDU19TVEFURV9ESVJgIHwgYC9jb250ZW50Ly5jb2RlLXNlcnZlcmAgfCBkYXRhLCBleHRlbnNpb25zLCBsb2dzLCBwaWRmaWxlcywgcGFzc3dvcmQgfAp8IGBDU19DQUNIRV9ESVJgIHwgYCRDU19TVEFURV9ESVIvY2FjaGVgIHwgcmVsZWFzZSB0YXJiYWxsIGNhY2hlIOKAlCBwb2ludCBhdCBEcml2ZSB0byBza2lwIHJlLWRvd25sb2FkcyB8CnwgYENTX1ZFUlNJT05gIHwgbGF0ZXN0IHwgcGluIGEgY29kZS1zZXJ2ZXIgdmVyc2lvbiB8CnwgYENTX1BSRUZJWGAgfCBgL3Vzci9sb2NhbGAgfCBpbnN0YWxsIHByZWZpeCB8CnwgYENTX0VYVEVOU0lPTlNgIHwg4oCUIHwgY29tbWEtc2VwYXJhdGVkIGV4dGVuc2lvbiBJRHMsIGluc3RhbGxlZCBvbmNlIHwKfCBgQ1NfSURMRV9USU1FT1VUYCB8IGAxODAwYCB8IHN1cGVydmlzZTogaWRsZSBzZWNvbmRzIGJlZm9yZSBzaHV0ZG93biB8CnwgYENTX01BWF9SVU5USU1FYCB8IGAzOTYwMGAgfCBzdXBlcnZpc2U6IHRvdGFsIHNlY29uZHMgYmVmb3JlIHNodXRkb3duIHwKfCBgQ1NfTUFYX1JFU1RBUlRTYCB8IGAyMGAgfCBzdXBlcnZpc2U6IGNvbnNlY3V0aXZlIGZhaWx1cmVzIGJlZm9yZSBnaXZpbmcgdXAgfAp8IGBDU19BTExPV19JTlNFQ1VSRWAgfCBgMGAgfCBtdXN0IGJlIGAxYCB0byBhY2NlcHQgYC0tbm8tYXV0aGAgfAp8IGBDU19TVEFURV9ESVJgIG9uIERyaXZlIHwg4oCUIHwgYENTX1NUQVRFX0RJUj0vY29udGVudC9kcml2ZS9NeURyaXZlLy5jc2Aga2VlcHMgZXh0ZW5zaW9ucyBhY3Jvc3MgcnVudGltZXMgYXQgc29tZSBGUyBsYXRlbmN5IGNvc3QgfAoKIyMgRGlhZ25vc3RpY3MKCmBgYGJhc2gKYmFzaCBjb2xhYi9kb2N0b3Iuc2gKYGBgCgpSZXBvcnRzIGFyY2hpdGVjdHVyZSwgT1MsIG1lbW9yeSwgYC9jb250ZW50YCBmcmVlIHNwYWNlLCBjb2RlLXNlcnZlciB2ZXJzaW9uLAp0dW5uZWwgY3JlZGVudGlhbHMsIGxpdmUgcHJvY2Vzc2VzLCBwb3J0IHN0YXRlLCBlc3RhYmxpc2hlZCBjb25uZWN0aW9ucywKYC9oZWFsdGh6YCwgbG9nIHNpemVzLiBSZWFkLW9ubHkuIFRoZSBwYXNzd29yZCBpcyBoaWRkZW4gaW4gaXRzIG91dHB1dDsgcmVhZApgJENTX1NUQVRFX0RJUi9jcmVkZW50aWFscy50eHRgIChtb2RlIDA2MDApIGlmIHlvdSBuZWVkIGl0LgoKIyMgV2h5IHRoZXNlIGZsYWdzCgp8IEZsYWcgfCBSZWFzb24gfAp8LS0tfC0tLXwKfCBgLS1iaW5kLWFkZHIgMTI3LjAuMC4xOlBPUlRgIHwgdGhlIHR1bm5lbCBhZ2VudCBpcyBpbiB0aGUgc2FtZSBWTTsgbm90aGluZyBuZWVkcyBhIHB1YmxpYyBiaW5kIHwKfCBgLS1hdXRoIHBhc3N3b3JkYCB8IHR1bm5lbCBob3N0bmFtZXMgZ2V0IHByb2JlZCB3aXRoaW4gbWludXRlcyDigJQgbmV2ZXIgYC0tYXV0aCBub25lYCB8CnwgYC0tZGlzYWJsZS10ZWxlbWV0cnlgIC8gYC0tZGlzYWJsZS11cGRhdGUtY2hlY2tgIHwgbm9pc2UsIHBsdXMgd3JpdGVzIHRvIGFuIGVwaGVtZXJhbCBkaXNrIHwKfCBgLS1kaXNhYmxlLXdvcmtzcGFjZS10cnVzdGAgfCBvdGhlcndpc2UgdGhlIHRydXN0IG1vZGFsIGVhdHMgdGhlIGZpcnN0IGNsaWNrIHwKfCBgLS11c2VyLWRhdGEtZGlyYCAvIGAtLWV4dGVuc2lvbnMtZGlyYCB8IGtlZXBzIHN0YXRlIGluIG9uZSBwZXJzaXN0YWJsZSBkaXJlY3RvcnkgfAp8IGAtLXRydXN0ZWQtb3JpZ2lucyAnKidgIHwgb25seSBpZiBhIHByb3h5ICoqcmV3cml0ZXMqKiBgSG9zdGAgKGAtLWhvc3QtaGVhZGVyPXJld3JpdGVgLCBDYWRkeSBgaGVhZGVyX3VwIEhvc3RgKS4gVGhhdCBpcyB0aGUgb25lIGNhc2UgdGhhdCBsb29rcyBsaWtlIGl0IG5lZWRzIGEgcGF0Y2ggYW5kIGRvZXMgbm90LiB8CgojIyBUcm91Ymxlc2hvb3RpbmcKCnwgU3ltcHRvbSB8IENhdXNlIHwKfC0tLXwtLS18CnwgYmxhbmsgcGFnZSwgbm8gZWRpdG9yIHwgbmdyb2sgd2FybmluZyBwYWdlIG5vdCBjbGlja2VkIHRocm91Z2ggfAp8IGBFUlJfTkdST0tfNDAxOGAgfCBubyBhdXRodG9rZW46IGBuZ3JvayBjb25maWcgYWRkLWF1dGh0b2tlbiA8dG9rZW4+YCB8CnwgYEVSUl9OR1JPS18xMDhgIHwgZnJlZSB0aWVyIGFsbG93cyBvbmUgYWdlbnQgc2Vzc2lvbjsgYGJhc2ggY29sYWIvc3RvcC5zaGAgZmlyc3QgfAp8IHN0YWxlIDQwNCBVUkwgfCBwcmV2aW91cyBydW50aW1lJ3MgVVJMOyByZS1ydW4gYHNlcnZlLnNoYCB8CnwgZWRpdG9yIGxvYWRzLCB0ZXJtaW5hbCBkaWVzIHwgV2ViU29ja2V0IG5vdCBjYXJyaWVkIGJ5IHRoZSBwYXRoIGluIHVzZSB8CnwgYGRpZTogcG9ydCA4MDgwIGluIHVzZWAgfCBvcnBoYW5lZCBwcm9jZXNzOyBgYmFzaCBjb2xhYi9zdG9wLnNoYCB8CnwgYGV4dGVuc2lvbnMgYWxyZWFkeSBpbnN0YWxsZWRgIGJ1dCBtaXNzaW5nIHwgZGVsZXRlIGAkQ1NfU1RBVEVfRElSLy5leHRlbnNpb25zLSpgIGFuZCByZXN0YXJ0IHwKCiMjIEhhcmRlbmluZyBub3RlcwoKKiBUaGUgcGFzc3dvcmQgZmlsZSBpcyBgMDYwMGA7IGBjcmVkZW50aWFscy50eHRgIGlzIGAwNjAwYCBhbmQgaG9sZHMgdGhlCiAgcGxhaW50ZXh0LiBGb3IgYSBzdHJvbmdlciBzZXR1cCB1c2UgY29kZS1zZXJ2ZXIncyBgSEFTSEVEX1BBU1NXT1JEYAogIChhcmdvbjJpZCkgaW4gYH4vLmNvbmZpZy9jb2RlLXNlcnZlci9jb25maWcueWFtbGAgYW5kIHNldCBgQ1NfUEFTU1dPUkRgIHRvCiAgbm90aGluZywgYWNjZXB0aW5nIHRoYXQgdGhlIHBhc3N3b3JkIGlzIHRoZW4gbm90IHJlY292ZXJhYmxlIGZyb20gZGlzay4KKiBgTkdST0tfQkFTSUNfQVVUSGAgcHV0cyBhIHNlY29uZCBjcmVkZW50aWFsIGluIGZyb250IG9mIGNvZGUtc2VydmVyIHNvIHBvcnQKICBzY2FubmVycyBuZXZlciByZWFjaCBpdC4gQ2hlYXAgYW5kIHdvcnRoIGl0IG9uIGEgZnJlZSBob3N0bmFtZS4KKiBOZXZlciBjb21iaW5lIGEgcHVibGljIHR1bm5lbCB3aXRoIGAtLW5vLWF1dGhgLCBhbmQgbmV2ZXIgYmluZCBgMC4wLjAuMGAgb24gYQogIGhvc3Qgd2l0aCBhIHB1YmxpYyBpbnRlcmZhY2UuCgojIyBPbiB0aGUgQ29sYWIgdGVybXMKCkNvbGFiJ3MgRkFRIGRvZXMgbm90IGxpc3QgcmVtb3RlLWRlc2t0b3Agb3IgcmVtb3RlLUlERSB1c2UgYXMgc3VwcG9ydGVkLiBUaGlzCndvcmtzLCBidXQgdGhlIHJ1bnRpbWUgaXMgZXBoZW1lcmFsLCB0aGUgVVJMIGNoYW5nZXMgb24gcmVzdGFydCB1bmxlc3MgeW91IHVzZSBhCm5hbWVkIHR1bm5lbCwgYW5kIGFjY291bnQtbGV2ZWwgdGhyb3R0bGluZyBpcyBhbHdheXMgcG9zc2libGUgcmVnYXJkbGVzcyBvZiBob3cKdGhlIHByb2Nlc3MgaXMgbmFtZWQgb3Igd3JhcHBlZC4K'
    ),
}

(BASE / "lib").mkdir(parents=True, exist_ok=True)
for rel, blob in FILES.items():
    f = BASE / rel
    f.write_bytes(base64.b64decode(blob))
    if f.suffix == ".sh":
        f.chmod(f.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print(f"wrote {len(FILES)} files to {BASE}")
for rel in sorted(FILES):
    print("  ", rel, (BASE / rel).stat().st_size, "bytes")

## 4. Install

Downloads code-server plus both tunnel agents. Install-time output is verbose on
purpose; if this cell fails, the traceback names the failing line.

In [ ]:
import os, subprocess

env = dict(os.environ, WITH_CLOUDFLARED="1")  # cloudflared is the no-account fallback
subprocess.run(["bash", str(BASE / "install.sh")], check=True, env=env)

## 5. Start

Starts the supervisor in the background, waits for the tunnel, and prints a
clickable link.

**ngrok users:** the first visit shows a "You are about to visit" page — click
through it, or the editor's assets load as HTML and you get a blank screen.

In [ ]:
import os, signal, time, pathlib, subprocess
from IPython.display import display, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# Re-running this cell must restart cleanly, so retire any previous supervisor
# (it holds a lock) and wait for it to let go of the port.
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print("stopped previous supervisor", pid)
        for _ in range(20):
            time.sleep(0.5)
            os.kill(pid, 0)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)
subprocess.run(["bash", str(BASE / "stop.sh")], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

creds = STATE / "credentials.txt"
creds.unlink(missing_ok=True)          # only accept a fresh write

out = STATE / "supervise.out"
fh = open(out, "w")
proc = subprocess.Popen(
    ["bash", str(BASE / "supervise.sh")],
    stdout=fh, stderr=subprocess.STDOUT,
    start_new_session=True,            # own process group, so stop can kill it cleanly
)
pid_file.write_text(str(proc.pid))

info = {}
for _ in range(240):
    time.sleep(1)
    if creds.exists():
        raw = creds.read_text().strip()
        if "url=" in raw:
            info = dict(l.split("=", 1) for l in raw.splitlines() if "=" in l)
            if info.get("url"):
                break
    if proc.poll() is not None:
        break

url, pw = info.get("url"), info.get("password")
if not url:
    print("startup did not finish. last lines of", out, "\n")
    print("\n".join(out.read_text().splitlines()[-40:]) if out.exists() else "(no log)")
else:
    display(HTML(
        f'<p style="font-size:17px;margin:8px 0"><b>code-server</b> &nbsp;'
        f'<a href="{url}" target="_blank">{url}</a></p>'
        f'<p style="margin:4px 0"><b>password</b> &nbsp;'
        f'<code style="font-size:15px">{pw}</code></p>'
    ))
    print(f"tunnel   : {info.get('tunnel')}")
    print(f"state    : {STATE}")
    print(f"supervisor pid {proc.pid} — restarts on crash, stops when idle for "
          f"{os.environ.get('CS_IDLE_TIMEOUT', 1800)}s")
    if info.get("tunnel") == "ngrok":
        print()
        print("ngrok free tier: click through the warning page before the editor loads.")


## 6. Status

In [ ]:
import pathlib
STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
supervise_log = STATE / "supervise.out"

if supervise_log.exists():
    print("── supervisor ─────────────────────────────────────────────")
    print("\n".join(supervise_log.read_text().splitlines()[-12:]))
    print()

!bash "{BASE}/doctor.sh"

## 7. Stop

In [ ]:
import os, pathlib, signal

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)   # supervisor was started detached
        print("stopped supervisor", pid)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)

!bash "{BASE}/stop.sh"

## Troubleshooting

| Symptom | Fix |
|---|---|
| blank page | click through the ngrok warning page, or set `TUNNEL = "cloudflared"` |
| URL 404s / changed | free ngrok rotates hostnames; set `NGROK_DOMAIN` to a reserved domain |
| cell 5 hangs then fails | `subprocess` raised — read the last lines it printed, the failing step is named |
| `port 8080 in use` | run cell 7, then cell 5 again |
| extensions missing | `rm -rf /content/.code-server/.extensions-*` and restart |
| editor loads, terminal dies | WebSocket not carried; switch tunnel type |

```bash
tail -n 60 /content/.code-server/code-server.log
tail -n 60 /content/.code-server/supervise.out
bash /content/colab-cs/doctor.sh
```

The readable sources are also in the repo at `colab/` — the embedded copies here
are generated from them, so prefer editing there and regenerating if you hack on
the scripts.